# STARFIRE Gap Streaming — OpenMC Reproduction
Reproduce Halley & Miller (1986), *Neutron Streaming Through Gaps in Fusion Reactor Shielding*.

**Run order — every cell, every session, top to bottom. Never skip.**

| Step | Cell | What it does |
|------|------|--------------|
| 1 | A | Install OpenMC (~15 min first run; ~30 sec cached) |
| 2 | B | Verify import |
| 3 | B10 | Build B-10 cross-section data |
| 4 | C | Download nuclear data (one-time ~15 min; skips if on Drive) |
| 5 | D | Smoke test (end-to-end check) |
| 6 | 1 | Configuration (GAP_TYPE, GAP_WIDTH_CM, etc.) |
| 7 | **2** | **Materials** — defines hfs_mat, mfs_mat, lfs_mat |
| 8 | **3** | **Geometry** — defines universe, geometry, det_cell |
| 9 | 3a | Geometry plot (visual check) |
| 10 | **4** | **Source** — 35-group degraded spectrum, forward hemisphere |
| 11 | **5** | **Tallies** |
| 12 | **6** | **Run** |

> After any kernel restart, re-run ALL steps in order — material IDs reset each session.  
> Never re-run step 7 (Materials) without immediately re-running step 8 (Geometry) after it.


---
## Cell A — Install OpenMC
First run: builds from C++ source (~15 min). Subsequent runs: installs from cached wheel on Drive (~30 sec).  
Also mounts Drive and sets DATA_DIR for nuclear data.

In [ ]:
import os, glob
from google.colab import drive

!apt-get install -y libhdf5-dev cmake ninja-build -q

drive.mount("/content/drive")
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DATA_DIR        = "/content/drive/MyDrive/openmc_data/endfb80"
WHEEL_DIR       = "/content/drive/MyDrive/openmc_data/wheel"
SOLIB_CACHE     = "/content/drive/MyDrive/openmc_data/libopenmc.so"
EXEC_CACHE      = "/content/drive/MyDrive/openmc_data/openmc_exec"
OPENMC_LIBDIR   = "/usr/local/lib/python3.12/dist-packages/openmc/lib"
os.makedirs(DATA_DIR,  exist_ok=True)
os.makedirs(WHEEL_DIR, exist_ok=True)

if os.path.exists(SOLIB_CACHE) and os.path.exists(EXEC_CACHE):
    # Fast path (~30 sec)
    wheels = [f for f in os.listdir(WHEEL_DIR) if f.startswith("openmc") and f.endswith(".whl")]
    if wheels:
        !pip install {WHEEL_DIR}/{wheels[0]} -q
    else:
        !pip install openmc==0.15.3 -q
    os.makedirs(OPENMC_LIBDIR, exist_ok=True)
    !cp {SOLIB_CACHE} {OPENMC_LIBDIR}/libopenmc.so
    !cp {SOLIB_CACHE} /usr/local/lib/libopenmc.so && ldconfig
    !cp {EXEC_CACHE} /usr/local/bin/openmc && chmod +x /usr/local/bin/openmc
    print("Fast path done: libopenmc.so + openmc binary restored.")
else:
    # Full CMake build (~20 min, once)
    print("Cloning OpenMC v0.15.3...")
    !git clone --depth 1 --branch v0.15.3 https://github.com/openmc-dev/openmc.git /content/openmc_src -q
    print("CMake configure...")
    !cmake -S /content/openmc_src -B /content/openmc_src/build -DCMAKE_BUILD_TYPE=Release -DOPENMC_USE_OPENMP=OFF 2>&1 | tail -3
    print("Building (~15 min)...")
    !cmake --build /content/openmc_src/build -j2 2>&1 | tail -3
    !pip install /content/openmc_src -q
    # Find and cache libopenmc.so
    sos = [s for s in glob.glob("/content/openmc_src/build/**/*.so*", recursive=True)
           if "openmc" in os.path.basename(s).lower()]
    # Find and cache openmc executable
    exes = [e for e in glob.glob("/content/openmc_src/build/**/openmc", recursive=True)
            if os.path.isfile(e) and os.access(e, os.X_OK) and not e.endswith(".so")]
    print(f"Found .so: {sos}")
    print(f"Found exe: {exes}")
    if sos:
        os.makedirs(OPENMC_LIBDIR, exist_ok=True)
        !cp {sos[0]} {OPENMC_LIBDIR}/libopenmc.so && cp {sos[0]} {SOLIB_CACHE}
        !cp {sos[0]} /usr/local/lib/libopenmc.so && ldconfig
    if exes:
        !cp {exes[0]} /usr/local/bin/openmc && chmod +x /usr/local/bin/openmc
        !cp {exes[0]} {EXEC_CACHE}
    if not sos or not exes:
        print("Some artifacts missing -- listing build output:")
        !find /content/openmc_src/build -maxdepth 3 -name "openmc*" 2>/dev/null

print(f"Data dir: {DATA_DIR}")


---
## Cell B â€” Verify

In [ ]:
import openmc, sys
print(f"OpenMC {openmc.__version__}  |  Python {sys.version.split()[0]}")

---
## Cell C — Nuclear data (one-time, ~15 min)
Downloads ENDF/B-VIII.0 tapes **only for the isotopes we use** and converts them to HDF5.  
Output is ~200–400 MB — fits in 3 GB Drive. Skips automatically on subsequent sessions.

In [ ]:
import os, zipfile, shutil

if "DATA_DIR" not in dir():
    from google.colab import drive; drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/openmc_data/endfb80"
WHEEL_DIR  = "/content/drive/MyDrive/openmc_data/wheel"
NJOY_CACHE = "/content/drive/MyDrive/openmc_data/njoy"

try:
    import openmc
except ModuleNotFoundError:
    !apt-get install -y libhdf5-dev -q
    wheels = [f for f in os.listdir(WHEEL_DIR) if f.startswith("openmc") and f.endswith(".whl")]
    !pip install {WHEEL_DIR}/{wheels[0]} -q
    import openmc

XS_FILE = f"{DATA_DIR}/cross_sections.xml"
os.environ["OPENMC_CROSS_SECTIONS"] = XS_FILE

if os.path.exists(XS_FILE) and os.path.getsize(XS_FILE) > 2000:
    print("Nuclear data already present.")
    print(XS_FILE)
else:
    # ── Step 1: NJOY binary ───────────────────────────────────────────
    import shutil as _sh
    if _sh.which("njoy") or _sh.which("njoy21"):
        print("NJOY already in PATH.")
    elif os.path.exists(NJOY_CACHE):
        print("Restoring NJOY from Drive cache...")
        !cp {NJOY_CACHE} /usr/local/bin/njoy && chmod +x /usr/local/bin/njoy
    else:
        print("Building NJOY2016 from source (~5 min)...")
        !apt-get install -y gfortran cmake -q
        !git clone --depth 1 https://github.com/njoy/NJOY2016.git /content/NJOY2016 -q
        !cmake -S /content/NJOY2016 -B /content/NJOY2016/build -DCMAKE_BUILD_TYPE=Release
        !cmake --build /content/NJOY2016/build -j2
        import glob; bins = glob.glob("/content/NJOY2016/build/**/njoy", recursive=True)
        if not bins: raise RuntimeError("njoy binary not found -- check cmake output")
        !cp {bins[0]} /usr/local/bin/njoy
        !cp /usr/local/bin/njoy {NJOY_CACHE}
        print("NJOY built and cached to Drive.")

    # ── Step 2: Download ENDF zip ─────────────────────────────────────
    ENDF_FILES = [
        "n-001_H_001.endf", "n-001_H_002.endf",
        "n-005_B_010.endf", "n-005_B_011.endf",
        "n-006_C_012.endf", "n-006_C_013.endf",
        "n-008_O_016.endf", "n-008_O_017.endf",
        "n-014_Si_028.endf", "n-014_Si_029.endf", "n-014_Si_030.endf",
        "n-022_Ti_046.endf", "n-022_Ti_047.endf", "n-022_Ti_048.endf",
        "n-022_Ti_049.endf", "n-022_Ti_050.endf",
        "n-024_Cr_050.endf", "n-024_Cr_052.endf", "n-024_Cr_053.endf", "n-024_Cr_054.endf",
        "n-025_Mn_055.endf",
        "n-026_Fe_054.endf", "n-026_Fe_056.endf", "n-026_Fe_057.endf", "n-026_Fe_058.endf",
        "n-028_Ni_058.endf", "n-028_Ni_060.endf", "n-028_Ni_061.endf",
        "n-028_Ni_062.endf", "n-028_Ni_064.endf",
        "n-042_Mo_092.endf", "n-042_Mo_094.endf", "n-042_Mo_095.endf",
        "n-042_Mo_096.endf", "n-042_Mo_097.endf", "n-042_Mo_098.endf", "n-042_Mo_100.endf",
    ]
    ZIP_URL    = "https://www.nndc.bnl.gov/endf-b8.0/zips/ENDF-B-VIII.0_neutrons.zip"
    ZIP_LOCAL  = "/content/endfb80_neutrons.zip"
    EXTRACT_DIR = "/content/endfb80_endf"

    if not os.path.exists(ZIP_LOCAL):
        print("Step 2: Downloading ENDF zip (~295 MB)...")
        !wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL}
    else:
        print("Step 2: Using existing zip.")

    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        all_names = z.namelist()
        for want in ENDF_FILES:
            hits = [n for n in all_names if want in n]
            if hits:
                z.extract(hits[0], EXTRACT_DIR)
            else:
                print(f"  NOT FOUND: {want}")

    # ── Step 3: NJOY processing ───────────────────────────────────────
    print("Step 3: Processing ENDF -> HDF5 via NJOY (~15 min)...")
    os.makedirs(DATA_DIR, exist_ok=True)
    lib = openmc.data.DataLibrary()
    ok, fail = 0, 0
    for root, _, files in os.walk(EXTRACT_DIR):
        for fname in sorted(files):
            if not fname.endswith(".endf"): continue
            path = os.path.join(root, fname)
            try:
                data = openmc.data.IncidentNeutron.from_njoy(path)
                h5 = os.path.join(DATA_DIR, f"{data.name}.h5")
                data.export_to_hdf5(h5, "w")
                lib.register_file(h5)
                print(f"  OK {data.name}")
                ok += 1
            except Exception as e:
                print(f"  FAIL {fname}: {e}")
                fail += 1

    lib.export_to_xml(XS_FILE)
    print(f"Done: {ok} OK, {fail} failed.")
    os.remove(ZIP_LOCAL)
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    print(XS_FILE)


---
## Cell D — Smoke test
Verifies OpenMC + nuclear data end-to-end before building the STARFIRE model.  
Uses only H-1 and O-16 (guaranteed to be in our data set) — no extra downloads.  
**Expected**: non-zero flux printed; assertion passes. If it fails, fix data path before continuing.

In [ ]:
import openmc, os, tempfile
import numpy as np

DATA_DIR = "/content/drive/MyDrive/openmc_data/endfb80"
os.environ["OPENMC_CROSS_SECTIONS"] = f"{DATA_DIR}/cross_sections.xml"

print("openmc binary:")
!which openmc
!openmc --version
print("libopenmc.so:")
!ls -lh /usr/local/lib/python3.12/dist-packages/openmc/lib/
print("xs xml:")
!ls -lh {DATA_DIR}/cross_sections.xml
print("h5 count:")
!ls {DATA_DIR}/*.h5 2>/dev/null | wc -l


with tempfile.TemporaryDirectory() as tmpdir:
    # Material: water
    mat = openmc.Material()
    mat.set_density('g/cm3', 1.0)
    mat.add_nuclide('H1',  2.0, 'ao')
    mat.add_nuclide('O16', 1.0, 'ao')

    # Geometry: 10x10x10 cm box, vacuum boundary
    xm = openmc.XPlane(-5, boundary_type='vacuum')
    xp = openmc.XPlane( 5, boundary_type='vacuum')
    ym = openmc.YPlane(-5, boundary_type='vacuum')
    yp = openmc.YPlane( 5, boundary_type='vacuum')
    zm = openmc.ZPlane(-5, boundary_type='vacuum')
    zp = openmc.ZPlane( 5, boundary_type='vacuum')
    cell = openmc.Cell(fill=mat, region=+xm & -xp & +ym & -yp & +zm & -zp)
    uni  = openmc.Universe(cells=[cell])

    # Source: 14.1 MeV point source at origin
    src = openmc.IndependentSource()
    src.space  = openmc.stats.Point((0, 0, 0))
    src.energy = openmc.stats.Discrete([14.1e6], [1.0])

    s = openmc.Settings()
    s.source    = src
    s.run_mode  = 'fixed source'
    s.batches   = 10
    s.particles = 1000

    tally = openmc.Tally()
    tally.filters = [openmc.CellFilter([cell])]
    tally.scores  = ['flux']

    model = openmc.Model(
        openmc.Geometry(uni),
        openmc.Materials([mat]),
        s,
        openmc.Tallies([tally]),
    )
    sp_path = model.run(cwd=tmpdir, output=True)
    sp = openmc.StatePoint(sp_path)
    flux = sp.get_tally().mean.flat[0]

assert flux > 0 and not np.isnan(flux), f'Bad flux: {flux}'
print(f'OK  flux = {flux:.3e} n/cm\u00b2 per source n')
print(f'OpenMC {openmc.__version__} + nuclear data working correctly.')

In [ ]:
# B10 via ENDF/B-VII.1  (NJOY2016 handles VII.1 reliably; no dlwh law=0 issue)
import openmc.data, os, shutil, zipfile, glob

DATA_DIR   = "/content/drive/MyDrive/openmc_data/endfb80"
NJOY_CACHE = "/content/drive/MyDrive/openmc_data/njoy"

if not shutil.which("njoy"):
    !cp {NJOY_CACHE} /usr/local/bin/njoy && chmod +x /usr/local/bin/njoy
    print("NJOY2016 restored.")

ZIP_URL_A = "https://www.nndc.bnl.gov/endf-b7.1/zips/ENDF-B-VII.1_neutrons.zip"
ZIP_URL_B = "https://www.nndc.bnl.gov/endf-b7.1/zips/ENDF-B-VII.1-neutrons.zip"
ZIP_LOCAL = "/content/endfb71_neutrons.zip"

if not os.path.exists(ZIP_LOCAL) or os.path.getsize(ZIP_LOCAL) < 1_000_000:
    print("Downloading ENDF/B-VII.1 neutrons (~200 MB)...")
    ret = os.system(f"wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL_A}")
    if ret != 0 or os.path.getsize(ZIP_LOCAL) < 1_000_000:
        print("Trying alternate URL...")
        os.system(f"wget -q --show-progress -O {ZIP_LOCAL} {ZIP_URL_B}")

EXTRACT_DIR = "/content/endfb71_b10"
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_LOCAL) as z:
    hits = [n for n in z.namelist() if "B_010" in n or "B-010" in n]
    print(f"B10 entries in zip: {hits}")
    for h in hits:
        z.extract(h, EXTRACT_DIR)

b10_files = [f for f in glob.glob(f"{EXTRACT_DIR}/**/*", recursive=True)
             if os.path.isfile(f) and ("B_010" in f or "B-010" in f)]
print(f"Extracted: {b10_files}")

if b10_files:
    print("Processing B10 from ENDF/B-VII.1 with NJOY2016 (stdout=True for diagnostics)...")
    try:
        data = openmc.data.IncidentNeutron.from_njoy(b10_files[0], temperatures=[293.6], stdout=True)
        h5 = f"{DATA_DIR}/B10.h5"
        data.export_to_hdf5(h5, "w")
        lib = openmc.data.DataLibrary.from_xml(f"{DATA_DIR}/cross_sections.xml")
        lib.register_file(h5)
        lib.export_to_xml(f"{DATA_DIR}/cross_sections.xml")
        print("B10 (VII.1) OK -- cross_sections.xml updated.")
        print("Now re-run Section 2 (B4C now uses natural boron).")
    except Exception as e:
        print(f"B10 VII.1 NJOY failed: {e}")
else:
    print("No B10 file extracted -- check zip URL or contents.")


---
## 1 â€” Configuration
Change `GAP_TYPE` and `GAP_WIDTH_CM` here; everything below regenerates.

In [ ]:
import os

# â”€â”€ Geometry parameters (from Fig. 2 of the paper) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
HFS_CM   = 50.0   # high-field-side shield thickness
MFS_CM   = 40.0   # mid-field-side shield thickness
LFS_CM   = 18.0   # low-field-side shield thickness
TOTAL_CM = HFS_CM + MFS_CM + LFS_CM   # 108 cm total

SLAB_HALF_WIDTH = 100.0   # half-width in x (200 cm wide slab)

# â”€â”€ Gap configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# GAP_TYPE: 'straight' or 'stepped'
GAP_TYPE      = 'straight'
GAP_WIDTH_CM  = 1.0    # try 1, 2, 5, 10

# For stepped gap: lateral shift per step segment
STEP_OFFSET_CM = 5.0

# â”€â”€ Run settings â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
BATCHES   = 200
PARTICLES = 50_000  # per batch; increase for better statistics

RUN_DIR = f'/content/run_{GAP_TYPE}_{GAP_WIDTH_CM}cm'
os.makedirs(RUN_DIR, exist_ok=True)
print(f"Output: {RUN_DIR}")
print(f"Shield: HFS {HFS_CM} + MFS {MFS_CM} + LFS {LFS_CM} = {TOTAL_CM} cm")
print(f"Gap: {GAP_TYPE}, {GAP_WIDTH_CM} cm wide")

---
## 2 â€” Materials
Verify compositions against Table II of the paper before final runs.

| Material | Density (g/cmÂ³) | Notes |
|---|---|---|
| TiHâ‚‚ | 3.75 | Verify from paper |
| Bâ‚„C | 2.52 | Natural boron |
| Fe-1422 | 8.0 | Treated as 316SS â€” **verify** |
| Hâ‚‚O | 1.0 | Available if paper includes water layer |
| void | ~0 | Gap interior |

In [ ]:
import openmc, os
os.environ['OPENMC_CROSS_SECTIONS'] = f'{DATA_DIR}/cross_sections.xml'

# ── Pure constituents ──────────────────────────────────────────────────────────
# All add_element() calls replaced with dominant isotopes only (library has
# ENDF/B-VII.1 dominant nuclides; minor isotopes O17/O18/Al27/etc. absent).

ti6al4v = openmc.Material(name='Ti6Al4V')
ti6al4v.set_density('g/cm3', 4.43)
ti6al4v.add_nuclide('Ti46', 0.0825, 'ao')
ti6al4v.add_nuclide('Ti47', 0.0744, 'ao')
ti6al4v.add_nuclide('Ti48', 0.7372, 'ao')
ti6al4v.add_nuclide('Ti49', 0.0541, 'ao')
ti6al4v.add_nuclide('Ti50', 0.0518, 'ao')
# Al (6 wt%) and V (4 wt%) not in library; negligible at 5 vol% in HFS

tih2 = openmc.Material(name='TiH2')
tih2.set_density('g/cm3', 3.75)
tih2.add_nuclide('Ti46', 0.0825 / 3.0, 'ao')
tih2.add_nuclide('Ti47', 0.0744 / 3.0, 'ao')
tih2.add_nuclide('Ti48', 0.7372 / 3.0, 'ao')
tih2.add_nuclide('Ti49', 0.0541 / 3.0, 'ao')
tih2.add_nuclide('Ti50', 0.0518 / 3.0, 'ao')
tih2.add_nuclide('H1',   2.0 / 3.0,    'ao')   # TiH2: 1 Ti + 2 H atoms

b4c = openmc.Material(name='B4C')
b4c.set_density('g/cm3', 2.52)
b4c.add_nuclide('B10', 4 * 0.199, 'ao')
b4c.add_nuclide('B11', 4 * 0.801, 'ao')
b4c.add_nuclide('C12', 1.0,       'ao')   # C13 (1.1%) absent from library

water = openmc.Material(name='H2O')
water.set_density('g/cm3', 1.0)
water.add_nuclide('H1',  2.0, 'ao')
water.add_nuclide('O16', 1.0, 'ao')       # O17/O18 absent from library

# Fe-1422 placeholder (316SS-like; dominant isotopes only)
fe1422 = openmc.Material(name='Fe1422')
fe1422.set_density('g/cm3', 8.0)
fe1422.add_nuclide('Fe56', 0.655, 'wo')   # 91.75% of natural Fe
fe1422.add_nuclide('Cr52', 0.170, 'wo')   # 83.79% of natural Cr
fe1422.add_nuclide('Ni58', 0.120, 'wo')   # 68.08% of natural Ni
fe1422.add_nuclide('Mn55', 0.025, 'wo')   # 100%  of natural Mn (merged Mo+Mn)
fe1422.add_nuclide('C12',  0.001, 'wo')
fe1422.add_nuclide('Si28', 0.029, 'wo')   # 92.23% of Si (merged Si+Mo remainder)
# Mo (2.5%) and minor isotopes dropped; Fe1422 is already a placeholder

# ── Homogenized shield zones (volume fractions, thesis §IID / Table III-1) ────
hfs_mat = openmc.Material.mix_materials(
    [ti6al4v, tih2, b4c, water], [0.05, 0.65, 0.15, 0.15], 'vo', name='HFS')

mfs_mat = openmc.Material.mix_materials(
    [fe1422, b4c, water], [0.70, 0.15, 0.15], 'vo', name='MFS')

lfs_mat = fe1422

void = openmc.Material(name='void')
void.set_density('g/cm3', 1e-10)
void.add_nuclide('H1', 1.0, 'ao')

materials = openmc.Materials([hfs_mat, mfs_mat, lfs_mat, void])
materials.export_to_xml(path=f'{RUN_DIR}/materials.xml')
print("Materials written.")
print(f"  HFS density = {hfs_mat.get_mass_density():.3f} g/cm3")
print(f"  MFS density = {mfs_mat.get_mass_density():.3f} g/cm3")
print(f"  LFS density = {lfs_mat.density:.3f} g/cm3")


---
## 3 — Geometry
> **Must run after Section 2 and before Section 3a.**  
> Defines `universe`, `geometry`, `det_cell`, `hfs_cell`, `mfs_cell`, `lfs_cell`.  
> Expected output: `Geometry written. Gap type=straight, width=1.0 cm`

Coordinate system:
- **z**: beam axis (source → shield → detector)
- **x**: transverse (gap width dimension)
- **y**: height (reflective BC → infinite slab)

Shield layers (M1 homogenized mixes):
```
z=0        z=50       z=90       z=108
 |-- HFS --|-- MFS --|-- LFS --|
 | TiH2+   | Fe1422+ | Fe1422  |
 | B4C+H2O | B4C+H2O |         |
```
Source void: z = -20 to 0  
Detector void: z = 108 to 130  
Gap slot: x = ±gap_width/2, full z extent of shield


In [ ]:
import openmc

w = GAP_WIDTH_CM / 2.0

# â”€â”€ Axial (z) planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
z_src   = openmc.ZPlane(z0=-20.0, boundary_type='vacuum')
z0      = openmc.ZPlane(z0=0.0)
z_hm    = openmc.ZPlane(z0=HFS_CM)
z_ml    = openmc.ZPlane(z0=HFS_CM + MFS_CM)
z1      = openmc.ZPlane(z0=TOTAL_CM)
z_det   = openmc.ZPlane(z0=TOTAL_CM + 52.0, boundary_type='vacuum')   # M2: was +22; extended to z=160

# â”€â”€ Transverse / height planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
x_min   = openmc.XPlane(x0=-SLAB_HALF_WIDTH, boundary_type='reflective')
x_max   = openmc.XPlane(x0= SLAB_HALF_WIDTH, boundary_type='reflective')
y_min   = openmc.YPlane(y0=-50.0, boundary_type='reflective')
y_max   = openmc.YPlane(y0= 50.0, boundary_type='reflective')

# â”€â”€ Gap planes â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
x_gm    = openmc.XPlane(x0=-w)
x_gp    = openmc.XPlane(x0= w)

in_box  = +x_min & -x_max & +y_min & -y_max

if GAP_TYPE == 'straight':
    in_gap = +x_gm & -x_gp & +z0 & -z1

elif GAP_TYPE == 'stepped':
    off  = STEP_OFFSET_CM
    xp1m = openmc.XPlane(x0=-w);       xp1p = openmc.XPlane(x0=w)
    xp2m = openmc.XPlane(x0=-w+off);   xp2p = openmc.XPlane(x0=w+off)
    xp3m = openmc.XPlane(x0=-w+2*off); xp3p = openmc.XPlane(x0=w+2*off)
    seg1 = +xp1m & -xp1p & +z0   & -z_hm
    seg2 = +xp2m & -xp2p & +z_hm & -z_ml
    seg3 = +xp3m & -xp3p & +z_ml & -z1
    in_gap = seg1 | seg2 | seg3

elif GAP_TYPE == 'single_step':
    off = STEP_OFFSET_CM                        # 3.0 = Fig IV-8 c-to-c
    zc  = TOTAL_CM / 2.0                        # step mid-shield (z=54)
    zk  = openmc.ZPlane(z0=zc)
    zk0 = openmc.ZPlane(z0=zc - GAP_WIDTH_CM)  # connector gap-width thick in z
    xl1m, xl1p = openmc.XPlane(x0=-w),       openmc.XPlane(x0=w)         # leg1 x=0
    xl2m, xl2p = openmc.XPlane(x0=-w + off), openmc.XPlane(x0=w + off)  # leg2 x=off
    xcm,  xcp  = openmc.XPlane(x0=-w),       openmc.XPlane(x0=w + off)  # connector
    leg1   = +xl1m & -xl1p & +z0  & -zk0
    conn   = +xcm  & -xcp  & +zk0 & -zk
    leg2   = +xl2m & -xl2p & +zk  & -z1
    in_gap = leg1 | conn | leg2

else:
    raise ValueError(f"Unknown GAP_TYPE: {GAP_TYPE}")

# â”€â”€ Cells â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
src_cell = openmc.Cell(name='source_void',   fill=None,   region=in_box & +z_src & -z0)
hfs_cell = openmc.Cell(name='HFS',           fill=hfs_mat, region=in_box & +z0   & -z_hm & ~in_gap)
mfs_cell = openmc.Cell(name='MFS',           fill=mfs_mat, region=in_box & +z_hm & -z_ml & ~in_gap)
lfs_cell = openmc.Cell(name='LFS',           fill=lfs_mat, region=in_box & +z_ml & -z1   & ~in_gap)
gap_cell = openmc.Cell(name='gap',           fill=void,   region=in_box & in_gap)
det_cell = openmc.Cell(name='detector_void', fill=None,   region=in_box & +z1   & -z_det)

universe = openmc.Universe(cells=[src_cell, hfs_cell, mfs_cell, lfs_cell, gap_cell, det_cell])
geometry = openmc.Geometry(universe)
geometry.export_to_xml(path=f'{RUN_DIR}/geometry.xml')
print(f"Geometry written.  Gap type={GAP_TYPE}, width={GAP_WIDTH_CM} cm")


---
## 3a â€” Geometry plot
Visual check before the first run. Look for:
- Three distinct shield layers (HFS / MFS / LFS) in different colours
- Narrow gap slot centred at x=0 running the full z extent of the shield
- Source void (z < 0) and detector void (z > 108) at top and bottom

If the gap is invisible or the layers are wrong, fix Section 3 before running.

In [ ]:
import openmc
import matplotlib.pyplot as plt

# Zoom width: show 20Ã— the gap width in x so even a 1 cm slot is visible
plot_width_x = max(GAP_WIDTH_CM * 20, 15.0)
plot_width_z = TOTAL_CM + 44.0   # includes source + detector void

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: zoomed in on gap region
universe.plot(
    basis='xz',
    origin=(0.0, 0.0, TOTAL_CM / 2),
    width=(plot_width_x, plot_width_z),
    pixels=(600, 600),
    color_by='material',
    axes=axes[0],
)
axes[0].set_title(f'Gap region (x \u00b1{plot_width_x/2:.0f} cm)')
axes[0].set_xlabel('x (cm)')
axes[0].set_ylabel('z (cm)')
# Layer boundaries
for z_line in [0, HFS_CM, HFS_CM + MFS_CM, TOTAL_CM]:
    axes[0].axhline(z_line - TOTAL_CM/2, color='white', lw=0.8, ls='--', alpha=0.7)

# Right: full slab width
universe.plot(
    basis='xz',
    origin=(0.0, 0.0, TOTAL_CM / 2),
    width=(SLAB_HALF_WIDTH * 2, plot_width_z),
    pixels=(600, 300),
    color_by='material',
    axes=axes[1],
)
axes[1].set_title('Full slab width (200 cm)')
axes[1].set_xlabel('x (cm)')

plt.suptitle(f'STARFIRE shield â€” {GAP_TYPE} gap, {GAP_WIDTH_CM} cm', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/geometry_xz.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {RUN_DIR}/geometry_xz.png")

---
## 4 — Source
**M1 patch:** 35-group degraded spectrum (Table III-3), forward hemisphere (+z).  
Replaces the original 14.1 MeV mono-directional beam.  
Group 1 (~14 MeV) carries 80.2% of the weight; thermal tail negligible.  
Expected output: `Settings written. 35-group source, forward hemisphere. 200 batches x 50,000 particles`


In [ ]:
import openmc
import numpy as np

# ── Thesis source spectrum ────────────────────────────────────────────────────
# Energies: per-group average energy, Table III-5 (MeV → eV).
# Weights:  normalized fraction, Table III-3 (sum = 1.0000; group 1 = 80.2% at ~14 MeV).
src_E_MeV = [
    14.209,    12.857,    11.107,    9.0940,    7.4453,    6.0957,    4.9907,
    4.0861,    3.3454,    2.7390,    2.2425,    1.8350,    1.5032,    1.2307,
    1.0076,    0.83496,   0.62031,   0.41580,   0.27872,   0.18684,   0.11824,
    5.9173e-2, 2.3431e-2, 1.0680e-2, 5.2282e-3, 2.4696e-3, 1.0193e-3, 2.7776e-4,
    6.1952e-5, 1.6640e-5, 7.8600e-6, 3.7130e-6, 1.7539e-6, 7.6970e-7, 2.5e-8,
]  # group 35 (thermal) representative value; weight negligible (0.014%)

src_w = [
    0.802160, 0.002043, 0.002307, 0.002108, 0.002043, 0.002121, 0.002068,
    0.002811, 0.003872, 0.005264, 0.006417, 0.007226, 0.007955, 0.007767,
    0.007689, 0.008817, 0.015311, 0.011663, 0.009942, 0.008022, 0.008884,
    0.012458, 0.007888, 0.007292, 0.006894, 0.006099, 0.009811, 0.009811,
    0.006961, 0.002446, 0.001697, 0.001067, 0.000605, 0.000341, 0.000144,
]
src_E_eV = [e * 1.0e6 for e in src_E_MeV]
assert len(src_E_eV) == len(src_w) == 35
assert abs(sum(src_w) - 1.0) < 1e-3, f"weights sum to {sum(src_w)}"

source = openmc.IndependentSource()
source.space = openmc.stats.Box(
    lower_left  = [-SLAB_HALF_WIDTH, -50.0, -10.0],
    upper_right = [ SLAB_HALF_WIDTH,  50.0,  -5.0],
)
# Forward hemisphere (+z): isotropic into the shield — lightweight proxy for
# the degraded angular flux at the shield jacket (resolved properly in M4).
source.angle = openmc.stats.PolarAzimuthal(
    mu  = openmc.stats.Uniform(0.0, 1.0),
    phi = openmc.stats.Uniform(0.0, 2.0 * np.pi),
    reference_uvw=(0.0, 0.0, 1.0),
)
source.energy = openmc.stats.Discrete(src_E_eV, src_w)

settings = openmc.Settings()
settings.source    = source
settings.run_mode  = 'fixed source'
settings.batches   = BATCHES
settings.particles = PARTICLES
settings.export_to_xml(path=f'{RUN_DIR}/settings.xml')
print(f"Settings written.  35-group source, forward hemisphere.  "
      f"{BATCHES} batches x {PARTICLES:,} particles")


---
## 5 â€” Tallies
Flux tally in the detector void cell, with energy binning.  
Dose is computed by convolving flux with ICRP-116 flux-to-dose coefficients.

In [ ]:
import openmc
import numpy as np

# ICRP-116 anterior-posterior neutron flux-to-dose (pSv·cm²)
dose_E, dose_C = openmc.data.dose_coefficients('neutron', 'AP')
dose_fn = openmc.EnergyFunctionFilter(dose_E, dose_C)

def plane_mesh(z_lo, z_hi):
    m = openmc.RegularMesh()
    m.lower_left  = [-10.0, -50.0, z_lo]   # x: -10..10, y: full height, z: thin slab
    m.upper_right = [ 10.0,  50.0, z_hi]
    m.dimension   = [40, 1, 1]              # 40 x-bins -> 0.5 cm each
    return m

# Detector planes: surface (thesis z=523 -> z=108),  +50 cm (thesis z=573 -> z=158)
surf_mesh = plane_mesh(TOTAL_CM,        TOTAL_CM + 1.0)    # 108-109 cm slab
far_mesh  = plane_mesh(TOTAL_CM + 49.5, TOTAL_CM + 50.5)   # 157.5-158.5 cm slab

def profile_tallies(tag, mesh):
    flux = openmc.Tally(name=f'flux_{tag}')
    flux.filters = [openmc.MeshFilter(mesh)]
    flux.scores  = ['flux']
    dose = openmc.Tally(name=f'dose_{tag}')
    dose.filters = [openmc.MeshFilter(mesh), dose_fn]
    dose.scores  = ['flux']
    return flux, dose

flux_surf, dose_surf = profile_tallies('surface', surf_mesh)
flux_far,  dose_far  = profile_tallies('50cm',    far_mesh)

# Keep scalar total for continuity with M1
total_tally = openmc.Tally(name='detector_total_flux')
total_tally.filters = [openmc.CellFilter([det_cell])]
total_tally.scores  = ['flux']

tallies = openmc.Tallies([flux_surf, dose_surf, flux_far, dose_far, total_tally])
tallies.export_to_xml(path=f'{RUN_DIR}/tallies.xml')
print("Tallies written: transverse flux+dose at surface (z=108) and +50 cm (z=158).")


---
## 6 â€” Run

In [ ]:
import openmc, os, shutil

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

# Guard: verify geometry references the same material objects as the current materials list.
# If this fires, re-run Section 3 (Geometry) immediately after Section 2 (Materials).
mat_ids_current = {m.id for m in materials}
for cell in geometry.root_universe.cells.values():
    fill = getattr(cell, 'fill', None)
    if fill is not None and hasattr(fill, 'id'):
        if fill.id not in mat_ids_current:
            raise RuntimeError(
                f"Section 6: geometry cell '{cell.name}' references material id={fill.id} "
                f"but current materials are {sorted(mat_ids_current)}. "
                f"Re-run Section 3 right after Section 2, then retry Section 6."
            )

# Clear stale HDF5 locks from any previous run
shutil.rmtree(RUN_DIR, ignore_errors=True)
os.makedirs(RUN_DIR)

# Re-export XMLs from in-scope objects
materials.export_to_xml(path=f"{RUN_DIR}/materials.xml")
geometry.export_to_xml(path=f"{RUN_DIR}/geometry.xml")
settings.export_to_xml(path=f"{RUN_DIR}/settings.xml")
tallies.export_to_xml(path=f"{RUN_DIR}/tallies.xml")

os.chdir(RUN_DIR)
openmc.run(threads=2, output=True)


In [ ]:
import openmc
import numpy as np
import matplotlib.pyplot as plt

sp_path_straight = sorted(__import__("glob").glob(f"{RUN_DIR}/statepoint.*.h5"))[-1]
sp = openmc.StatePoint(sp_path_straight)

x_edges   = np.linspace(-10.0, 10.0, 41)
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
CENSOR = 0.5   # mask bins with relative error above this

def profile(name):
    t    = sp.get_tally(name=name)
    mean = t.mean.flatten()
    sd   = t.std_dev.flatten()
    rel  = np.divide(sd, mean, out=np.full_like(mean, np.inf), where=mean > 0)
    return mean, rel

plt.figure(figsize=(8, 5))
for name, lbl, mk in [('dose_surface', 'z=523 (surface)', 'o'),
                      ('dose_50cm',    'z=573 (+50 cm)',  's')]:
    mean, rel = profile(name)
    good = rel < CENSOR
    line, = plt.semilogy(x_centers[good], mean[good], mk + '-', label=lbl)
    if (~good).any():
        plt.semilogy(x_centers[~good], mean[~good], mk, color=line.get_color(),
                     alpha=0.25, markerfacecolor='none')

plt.xlabel('Transverse distance  x (cm)')
plt.ylabel('Neutron dose  (per source n, ICRP-116 AP)')
plt.title('Straight 1 cm slot -- transverse dose profile\n'
          '(filled = measured, hollow = below sampling floor; cf. thesis Fig. IV-1/IV-2)')
plt.grid(True, which='both', alpha=0.3)
plt.legend()
plt.savefig(f'{RUN_DIR}/profile_straight_1cm.png', dpi=150, bbox_inches='tight')
plt.show()

# Numeric readout for Opus comparison
for name, lbl in [('dose_surface', 'surface'), ('dose_50cm', '+50 cm')]:
    mean, rel = profile(name)
    peak = mean[np.argmax(mean)]
    edge = mean[np.argmin(np.abs(x_centers - 5.0))]
    print(f"{lbl:8s}: peak={peak:.3e}  x~5cm={edge:.3e}  "
          f"falloff={np.log10(peak/edge):.1f} decades "
          f"(rel err @5cm={rel[np.argmin(np.abs(x_centers-5.0))]:.2f})")


---
## 6b — Solid-shield baseline (no gap)
Run once. Provides the denominator for the streaming enhancement ratio.
Expected: very low flux with B10 (may be at statistical noise floor).

In [ ]:
import openmc, os, shutil, glob

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
SOLID_DIR = '/content/run_solid'
shutil.rmtree(SOLID_DIR, ignore_errors=True)
os.makedirs(SOLID_DIR)

# Solid-shield geometry -- identical to gap case but no void slot
xmn = openmc.XPlane(x0=-SLAB_HALF_WIDTH, boundary_type='reflective')
xmx = openmc.XPlane(x0= SLAB_HALF_WIDTH, boundary_type='reflective')
ymn = openmc.YPlane(y0=-50.0, boundary_type='reflective')
ymx = openmc.YPlane(y0= 50.0, boundary_type='reflective')
zs  = openmc.ZPlane(z0=-20.0, boundary_type='vacuum')
z0s = openmc.ZPlane(z0=0.0)
zhs = openmc.ZPlane(z0=HFS_CM)
zms = openmc.ZPlane(z0=HFS_CM + MFS_CM)
z1s = openmc.ZPlane(z0=TOTAL_CM)
zds = openmc.ZPlane(z0=TOTAL_CM + 22.0, boundary_type='vacuum')

ib = +xmn & -xmx & +ymn & -ymx
src_s = openmc.Cell(fill=None,   region=ib & +zs  & -z0s)
hfs_s = openmc.Cell(fill=hfs_mat, region=ib & +z0s & -zhs)
mfs_s = openmc.Cell(fill=mfs_mat, region=ib & +zhs & -zms)
lfs_s = openmc.Cell(fill=lfs_mat, region=ib & +zms & -z1s)
det_s = openmc.Cell(fill=None,   region=ib & +z1s & -zds)

openmc.Geometry(openmc.Universe(cells=[src_s, hfs_s, mfs_s, lfs_s, det_s])).export_to_xml(path=f'{SOLID_DIR}/geometry.xml')
materials.export_to_xml(path=f'{SOLID_DIR}/materials.xml')
settings.export_to_xml(path=f'{SOLID_DIR}/settings.xml')

ft = openmc.Tally(name='detector_total_flux')
ft.filters = [openmc.CellFilter([det_s])]
ft.scores  = ['flux']
openmc.Tallies([ft]).export_to_xml(path=f'{SOLID_DIR}/tallies.xml')

os.chdir(SOLID_DIR)
openmc.run(threads=2, output=True)
print("Solid-shield run complete.")


---
## 7 â€” Results

In [ ]:
import openmc, glob
import numpy as np

DET_VOLUME = 200.0 * 100.0 * 22.0  # cm^3  (x: 200, y: 100, z: 22)

# --- Gap case ---
sp_gap = openmc.StatePoint(sorted(glob.glob(f'{RUN_DIR}/statepoint.*.h5'))[-1])
tg = sp_gap.get_tally(name='detector_total_flux')
gap_vi  = tg.mean.flat[0]               # n*cm per source n (volume-integrated)
gap_err = tg.std_dev.flat[0]
gap_avg = gap_vi / DET_VOLUME            # n/cm^2 per source n (average flux)

# --- Solid-shield case ---
solid_files = sorted(glob.glob('/content/run_solid/statepoint.*.h5'))
if solid_files:
    sp_solid = openmc.StatePoint(solid_files[-1])
    ts = sp_solid.get_tally(name='detector_total_flux')
    sol_vi  = ts.mean.flat[0]
    sol_err = ts.std_dev.flat[0]
    sol_avg = sol_vi / DET_VOLUME
    ratio   = gap_avg / sol_avg if sol_avg > 0 else float('inf')
else:
    sol_vi = sol_avg = sol_err = 0.0
    ratio  = float('inf')
    print("No solid-shield statepoint found -- run Section 6b first.")

# --- Dose (gap case, unnormalized) ---
td = sp_gap.get_tally(name='detector_dose')
dose_vi = td.mean.flat[0]

print(f"Detector volume: {DET_VOLUME:.0f} cm^3\n")
print(f"{'Case':<28} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}")
print("-" * 74)
print(f"{'Gap ' + GAP_TYPE + ' ' + str(GAP_WIDTH_CM) + ' cm':<28} {gap_vi:>16.4e} {gap_avg:>18.4e} {gap_err/gap_vi:>7.1%}")
if solid_files:
    rel = sol_err/sol_vi if sol_vi > 0 else float('nan')
    print(f"{'Solid shield (no gap)':<28} {sol_vi:>16.4e} {sol_avg:>18.4e} {rel:>7.1%}")
    print(f"\nStreaming enhancement (gap/solid):   {ratio:.2e}")
    print(f"Paper (1 cm straight slot):          ~1e+10  (Figs 3-9)")

print(f"\nDose (unnorm):  {dose_vi:.4e} pSv*cm^2 per source n")
print("\nNote: solid-shield flux near noise floor with B10 -- ratio is a lower bound.")
print("Variance reduction (CADIS/weight windows) needed for solid-shield accuracy.")


---
## 8 â€” Batch runner (all gap widths)
Sweeps all four straight-slot widths sequentially.

In [ ]:
import openmc, os, shutil, glob
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
results = {}

for gap_w in [1.0, 2.0, 5.0, 10.0]:
    run_dir = f'/content/run_straight_{gap_w}cm'
    # Clear stale HDF5 files from any previous run
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)

    ww = gap_w / 2.0
    xgm = openmc.XPlane(x0=-ww);  xgp = openmc.XPlane(x0=ww)
    ig  = +xgm & -xgp & +z0 & -z1
    ib  = +x_min & -x_max & +y_min & -y_max

    src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
    hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~ig)
    mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~ig)
    lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~ig)
    gap_c = openmc.Cell(fill=void,   region=ib & ig)
    det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

    openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
    materials.export_to_xml(path=f'{run_dir}/materials.xml')
    settings.export_to_xml(path=f'{run_dir}/settings.xml')

    ft = openmc.Tally(name='detector_total_flux')
    ft.filters = [openmc.CellFilter([det_c])]
    ft.scores  = ['flux']
    openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

    os.chdir(run_dir)
    openmc.run(threads=2, output=False)

    sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
    sp = openmc.StatePoint(sp_file)
    t  = sp.get_tally(name='detector_total_flux')
    results[gap_w] = (t.mean.flat[0], t.std_dev.flat[0])
    print(f"gap={gap_w:4.1f} cm  flux={t.mean.flat[0]:.4e} +/- {t.std_dev.flat[0]:.2e}  n*cm/source-n")

DET_VOLUME = 200.0 * 100.0 * 22.0
print(f"\n{'Gap (cm)':>10} {'Avg flux (n/cm2)':>20} {'Rel err':>10}")
print("-" * 44)
for gap_w, (mean, std) in sorted(results.items()):
    print(f"{gap_w:>10.1f} {mean/DET_VOLUME:>20.4e} {std/mean:>9.1%}")


---
## 8b — Batch runner: stepped configurations
Sweeps gap widths [1, 2, 5] cm x step offsets [2.5, 5, 10] cm = 9 runs.
Expected runtime: ~90 min total.

In [ ]:
import openmc, os, shutil, glob, json as _json
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DET_VOLUME   = 200.0 * 100.0 * 22.0
RESULTS_FILE = '/content/drive/MyDrive/openmc_data/stepped_results.json'
WIDTHS       = [1.0, 2.0, 5.0]
OFFSETS      = [2.5, 5.0, 10.0]

# Load any results already saved to Drive
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        saved = _json.load(f)
    results_stepped = {tuple(map(float, k.split(','))): v for k, v in saved.items()}
    print(f"Loaded {len(results_stepped)} saved results from Drive.")
else:
    results_stepped = {}

for gap_w in WIDTHS:
    for step_off in OFFSETS:
        key = (gap_w, step_off)
        if key in results_stepped:
            mean, std = results_stepped[key]
            print(f"SKIP  w={gap_w:.1f}cm off={step_off:.1f}cm  (already on Drive: {mean:.4e})")
            continue

        run_dir = f'/content/run_stepped_{gap_w}cm_{step_off}off'
        shutil.rmtree(run_dir, ignore_errors=True)
        os.makedirs(run_dir)

        w    = gap_w / 2.0
        xp1m = openmc.XPlane(x0=-w);            xp1p = openmc.XPlane(x0=w)
        xp2m = openmc.XPlane(x0=-w+step_off);   xp2p = openmc.XPlane(x0=w+step_off)
        xp3m = openmc.XPlane(x0=-w+2*step_off); xp3p = openmc.XPlane(x0=w+2*step_off)
        seg1   = +xp1m & -xp1p & +z0   & -z_hm
        seg2   = +xp2m & -xp2p & +z_hm & -z_ml
        seg3   = +xp3m & -xp3p & +z_ml & -z1
        in_gap = seg1 | seg2 | seg3
        ib     = +x_min & -x_max & +y_min & -y_max

        src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
        hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~in_gap)
        mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~in_gap)
        lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~in_gap)
        gap_c = openmc.Cell(fill=void,   region=ib & in_gap)
        det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

        openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
        materials.export_to_xml(path=f'{run_dir}/materials.xml')
        settings.export_to_xml(path=f'{run_dir}/settings.xml')
        ft = openmc.Tally(name='detector_total_flux')
        ft.filters = [openmc.CellFilter([det_c])]
        ft.scores  = ['flux']
        openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

        os.chdir(run_dir)
        openmc.run(threads=2, output=False)

        sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
        sp = openmc.StatePoint(sp_file)
        t  = sp.get_tally(name='detector_total_flux')
        mean, std = float(t.mean.flat[0]), float(t.std_dev.flat[0])
        results_stepped[key] = [mean, std]

        # Save to Drive after every run
        with open(RESULTS_FILE, 'w') as f:
            _json.dump({f'{k[0]},{k[1]}': v for k, v in results_stepped.items()}, f, indent=2)

        print(f"w={gap_w:.1f}cm off={step_off:.1f}cm  flux={mean:.4e} +/- {std:.2e}  ({std/mean:.1%})  [saved]")

print(f"\n{'Gap (cm)':>9} {'Offset (cm)':>12} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}")
print("-" * 68)
for (gap_w, step_off), (mean, std) in sorted(results_stepped.items()):
    print(f"{gap_w:>9.1f} {step_off:>12.1f} {mean:>16.4e} {mean/DET_VOLUME:>18.4e} {std/mean:>7.1%}")


---
## 9 — ML Surrogate (Gaussian Process)
Train a GP on the 13-point parametric dataset (4 straight + 9 stepped).  
Features: `gap_type`, `gap_width`, `step_offset`, `overlap_ratio` (derived).  
Target: `log₁₀(flux)` — working in log-space keeps the GP well-conditioned across 4 decades.

**Leave-one-out cross-validation** (13 folds) gives an honest estimate of predictive error with this small dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler

# ── Complete dataset ──────────────────────────────────────────────────────────
raw = [
    (0,  1.0,  0.0, 1.1123e-1),
    (0,  2.0,  0.0, 2.2145e-1),
    (0,  5.0,  0.0, 5.5724e-1),
    (0, 10.0,  0.0, 1.1243e+0),
    (1,  1.0,  2.5, 3.6202e-3),
    (1,  1.0,  5.0, 3.5729e-3),
    (1,  1.0, 10.0, 3.2396e-3),
    (1,  2.0,  2.5, 9.2993e-3),
    (1,  2.0,  5.0, 7.3288e-3),
    (1,  2.0, 10.0, 6.3946e-3),
    (1,  5.0,  2.5, 2.9945e-1),   # overlap regime
    (1,  5.0,  5.0, 4.1717e-2),
    (1,  5.0, 10.0, 2.0238e-2),
]

data = np.array([[r[0], r[1], r[2]] for r in raw], dtype=float)
flux = np.array([r[3] for r in raw])

# ── Feature engineering ───────────────────────────────────────────────────────
# log10(width): straight flux ∝ width exactly, so log(flux) is linear in
# log(width). The GP interpolates/extrapolates correctly in this coordinate.
log_width = np.log10(data[:, 1])

# overlap_ratio: fraction of gap with unobstructed line of sight
overlap = np.where(
    data[:, 0] == 0,
    1.0,
    np.maximum(0.0, 1.0 - 2.0 * data[:, 2] / data[:, 1])
)

X = np.column_stack([data[:, 0], log_width, data[:, 2], overlap])
y = np.log10(flux)

n      = len(y)
labels = [
    f"str {r[1]:.0f}cm" if r[0] == 0 else f"step {r[1]:.0f}cm/{r[2]:.0f}off"
    for r in raw
]
colors = ['#3b82f6' if r[0] == 0 else '#f59e0b' for r in raw]

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── GP — alpha jitter replaces WhiteKernel, no ConvergenceWarning ─────────────
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * RBF(length_scale=np.ones(X.shape[1]),
          length_scale_bounds=[(1e-2, 1e2)] * X.shape[1])
)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-8,
                               n_restarts_optimizer=10,
                               normalize_y=True, random_state=42)
gp.fit(X_scaled, y)
print(f"Full-data GP fitted.\nKernel: {gp.kernel_}\n")

# ── Leave-one-out cross-validation ────────────────────────────────────────────
y_pred_loo = np.zeros(n)
y_std_loo  = np.zeros(n)

for i in range(n):
    mask = np.ones(n, dtype=bool); mask[i] = False
    gp_i = GaussianProcessRegressor(kernel=kernel, alpha=1e-8,
                                     n_restarts_optimizer=5,
                                     normalize_y=True, random_state=42)
    gp_i.fit(X_scaled[mask], y[mask])
    mu, sig = gp_i.predict(X_scaled[[i]], return_std=True)
    y_pred_loo[i] = mu[0]
    y_std_loo[i]  = sig[0]

flux_true = 10 ** y
flux_pred = 10 ** y_pred_loo
flux_lo   = 10 ** (y_pred_loo - 2 * y_std_loo)
flux_hi   = 10 ** (y_pred_loo + 2 * y_std_loo)
err_dec   = y_pred_loo - y

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for ft, fp, fl, fh, lbl, c in zip(flux_true, flux_pred, flux_lo, flux_hi, labels, colors):
    ax.errorbar(ft, fp, yerr=[[fp - fl], [fh - fp]],
                fmt='o', color=c, capsize=4, markersize=7, label=lbl)
lo = min(flux_true.min(), flux_pred.min()) * 0.3
hi = max(flux_true.max(), flux_pred.max()) * 3.0
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='perfect')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo, hi);  ax.set_ylim(lo, hi)
ax.set_xlabel('OpenMC flux  (n·cm / source·n)')
ax.set_ylabel('GP prediction  (LOO)')
ax.set_title('LOO cross-validation — parity plot')
ax.legend(fontsize=7, ncol=2)
ax.text(0.03, 0.97, 'Error bars: ±2σ GP uncertainty', transform=ax.transAxes,
        fontsize=7, va='top', color='gray')

ax2 = axes[1]
ax2.barh(range(n), err_dec, color=colors)
ax2.axvline(0,    color='k',  lw=1)
ax2.axvline(-0.5, color='r',  lw=0.8, ls='--', label='±0.5 decade')
ax2.axvline( 0.5, color='r',  lw=0.8, ls='--')
ax2.set_yticks(range(n))
ax2.set_yticklabels(labels, fontsize=8)
ax2.set_xlabel('log₁₀(predicted / true)  [decades]')
ax2.set_title('LOO residuals  (0 = perfect)')
ax2.legend(fontsize=8)

plt.suptitle('STARFIRE GP Surrogate — LOO CV  (v3: log₁₀(width), alpha jitter)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/gp_loo.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Summary ───────────────────────────────────────────────────────────────────
abs_err = np.abs(err_dec)
print(f"LOO summary ({n} points, features: gap_type, log10_width, offset, overlap):")
print(f"  Mean |error|:      {abs_err.mean():.2f} decades")
print(f"  Max  |error|:      {abs_err.max():.2f} decades  ({labels[np.argmax(abs_err)]})")
print(f"  Within ±0.5 dec:   {(abs_err <= 0.5).sum()}/{n}")
print(f"  Within ±1.0 dec:   {(abs_err <= 1.0).sum()}/{n}")
print()
for lbl, ed in zip(labels, err_dec):
    bar = '█' * int(abs(ed) * 10)
    sign = '+' if ed >= 0 else '-'
    print(f"  {lbl:<22} {sign}{abs(ed):.2f} dec  {bar}")

# ── Predict new configurations ────────────────────────────────────────────────
def predict(gap_type, width, offset):
    ov = 1.0 if gap_type == 0 else max(0.0, 1.0 - 2 * offset / width)
    x  = scaler.transform([[gap_type, np.log10(width), offset, ov]])
    mu, sig = gp.predict(x, return_std=True)
    return 10**mu[0], 10**(mu[0] - 2*sig[0]), 10**(mu[0] + 2*sig[0])

print("\nExample predictions (full GP, not LOO):")
cases = [
    ("straight 3 cm",    0, 3.0, 0.0),
    ("straight 7 cm",    0, 7.0, 0.0),
    ("stepped 3cm/7off", 1, 3.0, 7.0),
    ("stepped 2cm/3off", 1, 2.0, 3.0),
    ("stepped 5cm/7off", 1, 5.0, 7.0),
]
for name, gt, w, off in cases:
    fp, fl, fh = predict(gt, w, off)
    print(f"  {name:<22}  flux = {fp:.3e}  [{fl:.2e}, {fh:.2e}]  (±2σ)")


---
## 10 — Targeted overlap-boundary runs
5 new Monte Carlo runs targeting the regime where the GP had its worst error (0.61 dec at w=5cm, off=2.5cm).
Covers: deep overlap, the exact boundary, and just past it, for both w=5cm and w=2/3cm.

Results saved to `openmc_data/overlap_results.json` on Drive (checkpoint/resume supported).  
Expected runtime: ~45 min total.

In [ ]:
import openmc, os, shutil, glob, json
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DET_VOLUME       = 200.0 * 100.0 * 22.0
NEW_RESULTS_FILE = '/content/drive/MyDrive/openmc_data/overlap_results.json'

# 5 targeted points around the overlap boundary (offset ≈ half-width)
# These fill the regime where the original GP had max error 0.61 dec
NEW_CASES = [
    (5.0,  1.0),   # w=5cm, off=1cm  — deep overlap (off < half-width)
    (5.0,  3.5),   # w=5cm, off=3.5cm — just past boundary
    (5.0,  4.0),   # w=5cm, off=4cm   — well past boundary
    (2.0,  1.0),   # w=2cm, off=1cm   — boundary for narrower gap
    (3.0,  1.5),   # w=3cm, off=1.5cm — new width, at boundary
]

if os.path.exists(NEW_RESULTS_FILE):
    with open(NEW_RESULTS_FILE) as f:
        results_new = {tuple(map(float, k.split(','))): v for k, v in json.load(f).items()}
    print(f"Loaded {len(results_new)} existing results from Drive.")
else:
    results_new = {}

for gap_w, step_off in NEW_CASES:
    key = (gap_w, step_off)
    if key in results_new:
        mean, std = results_new[key]
        print(f"SKIP  w={gap_w:.1f}cm off={step_off:.1f}cm  (already done: {mean:.4e})")
        continue

    run_dir = f'/content/run_overlap_{gap_w}cm_{step_off}off'
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)

    w    = gap_w / 2.0
    xp1m = openmc.XPlane(x0=-w);            xp1p = openmc.XPlane(x0=w)
    xp2m = openmc.XPlane(x0=-w+step_off);   xp2p = openmc.XPlane(x0=w+step_off)
    xp3m = openmc.XPlane(x0=-w+2*step_off); xp3p = openmc.XPlane(x0=w+2*step_off)
    seg1   = +xp1m & -xp1p & +z0   & -z_hm
    seg2   = +xp2m & -xp2p & +z_hm & -z_ml
    seg3   = +xp3m & -xp3p & +z_ml & -z1
    in_gap = seg1 | seg2 | seg3
    ib     = +x_min & -x_max & +y_min & -y_max

    src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
    hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~in_gap)
    mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~in_gap)
    lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~in_gap)
    gap_c = openmc.Cell(fill=void,   region=ib & in_gap)
    det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

    openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
    materials.export_to_xml(path=f'{run_dir}/materials.xml')
    settings.export_to_xml(path=f'{run_dir}/settings.xml')

    ft = openmc.Tally(name='detector_total_flux')
    ft.filters = [openmc.CellFilter([det_c])]
    ft.scores  = ['flux']
    openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

    os.chdir(run_dir)
    openmc.run(threads=2, output=False)

    sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
    sp = openmc.StatePoint(sp_file)
    t  = sp.get_tally(name='detector_total_flux')
    mean, std = float(t.mean.flat[0]), float(t.std_dev.flat[0])
    results_new[key] = [mean, std]

    with open(NEW_RESULTS_FILE, 'w') as f:
        json.dump({f'{k[0]},{k[1]}': v for k, v in results_new.items()}, f, indent=2)

    print(f"w={gap_w:.1f}cm off={step_off:.1f}cm  flux={mean:.4e} ± {std:.2e}  ({std/mean:.1%})  [saved]")

print(f"\n{'Gap (cm)':>9} {'Offset (cm)':>12} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}")
print("-" * 68)
for (gap_w, step_off), (mean, std) in sorted(results_new.items()):
    print(f"{gap_w:>9.1f} {step_off:>12.1f} {mean:>16.4e} {mean/DET_VOLUME:>18.4e} {std/mean:>7.1%}")

print("\nDone. Run Section 11 to retrain the GP with these new points.")


---
## 11 — Two-GP Surrogate (18 points)
Separate GPs for straight and stepped slots — eliminates the mixed categorical/continuous
kernel problem that caused ConvergenceWarnings and the `str 10cm` LOO artifact in v1.

- **Straight GP**: 4 points, 1D feature: `log₁₀(width)` — clean power-law fit
- **Stepped GP**: 14 points, 3D features: `log₁₀(width)`, `offset`, `overlap_ratio`

Run after Section 10 has saved `overlap_results.json` to Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, os, warnings
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.preprocessing import StandardScaler

# ── Mount Drive if needed ─────────────────────────────────────────────────────
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# ── Dataset ───────────────────────────────────────────────────────────────────
raw_straight = [
    (1.0,  1.1123e-1),
    (2.0,  2.2145e-1),
    (5.0,  5.5724e-1),
    (10.0, 1.1243e+0),
]

raw_stepped = [
    (1.0,  2.5, 3.6202e-3),
    (1.0,  5.0, 3.5729e-3),
    (1.0, 10.0, 3.2396e-3),
    (2.0,  2.5, 9.2993e-3),
    (2.0,  5.0, 7.3288e-3),
    (2.0, 10.0, 6.3946e-3),
    (5.0,  2.5, 2.9945e-1),
    (5.0,  5.0, 4.1717e-2),
    (5.0, 10.0, 2.0238e-2),
]

NEW_RESULTS_FILE = '/content/drive/MyDrive/openmc_data/overlap_results.json'
if os.path.exists(NEW_RESULTS_FILE):
    with open(NEW_RESULTS_FILE) as f:
        for k, (mean, _) in json.load(f).items():
            gap_w, step_off = map(float, k.split(','))
            raw_stepped.append((gap_w, step_off, mean))
    print(f"Section 10 points loaded.")
else:
    print("WARNING: overlap_results.json not found — run Section 10 first.")

GAP_RESULTS_FILE = '/content/drive/MyDrive/openmc_data/gap_results.json'
if os.path.exists(GAP_RESULTS_FILE):
    with open(GAP_RESULTS_FILE) as f:
        for k, (mean, _) in json.load(f).items():
            gap_w, step_off = map(float, k.split(','))
            raw_stepped.append((gap_w, step_off, mean))
    print(f"Section 12 points loaded.")

print(f"Total stepped dataset: {len(raw_stepped)} points")

# ── Straight GP (1D: log10_width) ─────────────────────────────────────────────
ws  = np.array([r[0] for r in raw_straight])
fs  = np.array([r[1] for r in raw_straight])
Xs  = np.log10(ws).reshape(-1, 1)
ys  = np.log10(fs)

scaler_s = StandardScaler()
Xs_sc    = scaler_s.fit_transform(Xs)

k_s  = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))
gp_s = GaussianProcessRegressor(k_s, alpha=1e-8, n_restarts_optimizer=10,
                                 normalize_y=True, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    gp_s.fit(Xs_sc, ys)
print(f"Straight GP kernel: {gp_s.kernel_}")

loo_s = np.zeros(len(ys))
for i in range(len(ys)):
    mask = np.ones(len(ys), bool); mask[i] = False
    g = GaussianProcessRegressor(k_s, alpha=1e-8, n_restarts_optimizer=5,
                                 normalize_y=True, random_state=42)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        g.fit(Xs_sc[mask], ys[mask])
    loo_s[i] = g.predict(Xs_sc[[i]])[0]

# ── Stepped GP (3D: log10_width, offset, norm_offset=2*off/width) ──────────────
wp  = np.array([r[0] for r in raw_stepped])
op  = np.array([r[1] for r in raw_stepped])
fp  = np.array([r[2] for r in raw_stepped])
ovr = 2.0 * op / wp  # normalized offset: boundary at 1.0 for all widths

Xp  = np.column_stack([np.log10(wp), op, ovr])
yp  = np.log10(fp)
np_ = len(yp)

scaler_p = StandardScaler()
Xp_sc    = scaler_p.fit_transform(Xp)

k_p  = (ConstantKernel(1.0, (1e-3, 1e3))
        * RBF(np.ones(3), [(1e-2, 1e2)] * 3))
gp_p = GaussianProcessRegressor(k_p, alpha=1e-8, n_restarts_optimizer=10,
                                 normalize_y=True, random_state=42)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    gp_p.fit(Xp_sc, yp)
print(f"Stepped  GP kernel: {gp_p.kernel_}\n")

loo_p     = np.zeros(np_)
loo_p_std = np.zeros(np_)
for i in range(np_):
    mask = np.ones(np_, bool); mask[i] = False
    g = GaussianProcessRegressor(k_p, alpha=1e-8, n_restarts_optimizer=5,
                                 normalize_y=True, random_state=42)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        g.fit(Xp_sc[mask], yp[mask])
    mu, sig = g.predict(Xp_sc[[i]], return_std=True)
    loo_p[i] = mu[0]; loo_p_std[i] = sig[0]

# ── Summary ───────────────────────────────────────────────────────────────────
err_s = loo_s - ys
err_p = loo_p - yp
all_err = np.concatenate([err_s, err_p])
abs_all = np.abs(all_err)

labels_s = [f"str {w:.0f}cm" for w in ws]
labels_p = [f"step {r[0]:.0f}cm/{r[1]:.1f}off" for r in raw_stepped]

print(f"═══ Straight GP LOO (4 points) ═══")
for lbl, e in zip(labels_s, err_s):
    bar = '█' * int(abs(e) * 10)
    print(f"  {lbl:<22} {'+' if e>=0 else '-'}{abs(e):.2f} dec  {bar}")
print(f"  Mean |err|: {np.abs(err_s).mean():.2f}  Max: {np.abs(err_s).max():.2f}\n")

print(f"═══ Stepped GP LOO ({np_} points) ═══")
for lbl, e in zip(labels_p, err_p):
    bar = '█' * int(abs(e) * 10)
    print(f"  {lbl:<26} {'+' if e>=0 else '-'}{abs(e):.2f} dec  {bar}")
print(f"  Mean |err|: {np.abs(err_p).mean():.2f}  Max: {np.abs(err_p).max():.2f}\n")

print(f"═══ Combined ({len(all_err)} points) ═══")
print(f"  Mean |error|:      {abs_all.mean():.2f} decades")
print(f"  Max  |error|:      {abs_all.max():.2f} decades")
print(f"  Within ±0.5 dec:   {(abs_all <= 0.5).sum()}/{len(all_err)}")
print(f"  Within ±1.0 dec:   {(abs_all <= 1.0).sum()}/{len(all_err)}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
# Straight
ft_s = 10**ys; fp_s = 10**loo_s
ax.scatter(ft_s, fp_s, color='#3b82f6', s=80, zorder=5, label='straight (4 pts)')
# Stepped
ft_p = 10**yp; fp_p_arr = 10**loo_p
fp_lo = 10**(loo_p - 2*loo_p_std); fp_hi = 10**(loo_p + 2*loo_p_std)
colors_p = ['#ef4444' if r[1] in [1.0, 3.5, 4.0, 1.5] else '#f59e0b' for r in raw_stepped]
for ft, fp_v, fl, fh, c in zip(ft_p, fp_p_arr, fp_lo, fp_hi, colors_p):
    ax.errorbar(ft, fp_v, yerr=[[fp_v-fl],[fh-fp_v]], fmt='o', color=c,
                capsize=4, markersize=7)
ax.scatter([], [], color='#f59e0b', s=60, label='stepped orig (9 pts)')
ax.scatter([], [], color='#ef4444', s=60, label='stepped new (5 pts)')

lo = min(ft_s.min(), ft_p.min()) * 0.3
hi = max(ft_s.max(), ft_p.max()) * 3.0
ax.plot([lo,hi],[lo,hi],'k--',lw=1,label='perfect')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(lo,hi); ax.set_ylim(lo,hi)
ax.set_xlabel('OpenMC flux  (n·cm / source·n)')
ax.set_ylabel('GP prediction  (LOO)')
ax.set_title('Two-GP LOO — parity plot')
ax.legend(fontsize=8)

ax2 = axes[1]
all_labels = labels_s + labels_p
all_colors = ['#3b82f6']*4 + colors_p
ax2.barh(range(len(all_err)), all_err, color=all_colors)
ax2.axvline(0,    color='k', lw=1)
ax2.axvline(-0.5, color='r', lw=0.8, ls='--', label='±0.5 dec')
ax2.axvline( 0.5, color='r', lw=0.8, ls='--')
ax2.set_yticks(range(len(all_err)))
ax2.set_yticklabels(all_labels, fontsize=7)
ax2.set_xlabel('log₁₀(predicted / true)')
ax2.set_title('LOO residuals')
ax2.legend(fontsize=8)

plt.suptitle('STARFIRE Two-GP Surrogate — LOO CV  (norm_offset feature)', fontsize=13)
plt.tight_layout()
plt.savefig('/content/gp_two_model_loo_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Prediction helper ─────────────────────────────────────────────────────────
def predict(gap_type, width, offset=0.0):
    if gap_type == 'straight':
        x = scaler_s.transform([[np.log10(width)]])
        mu, sig = gp_s.predict(x, return_std=True)
    else:
        ov = 2.0 * offset / width  # normalized offset
        x  = scaler_p.transform([[np.log10(width), offset, ov]])
        mu, sig = gp_p.predict(x, return_std=True)
    return 10**mu[0], 10**(mu[0]-2*sig[0]), 10**(mu[0]+2*sig[0])

print("\nExample predictions:")
for desc, gt, w, off in [
    ("straight 3cm",    'straight', 3.0, 0.0),
    ("straight 7cm",    'straight', 7.0, 0.0),
    ("stepped 3cm/7off",'stepped',  3.0, 7.0),
    ("stepped 2cm/3off",'stepped',  2.0, 3.0),
    ("stepped 5cm/3off",'stepped',  5.0, 3.0),
]:
    fp_v, fl, fh = predict(gt, w, off)
    print(f"  {desc:<22}  {fp_v:.3e}  [{fl:.2e}, {fh:.2e}]")


---
## 12 — Gap-closing runs (3 points)
Targeted at the `1cm/10off` LOO outlier (1.07 dec in Section 11).
Root cause: no data at norm_offset > 10 for any width — GP extrapolates blindly.

New runs:
- w=1cm, off=7.5cm (norm_off=15) — brackets 1cm/10off directly
- w=3cm, off=5.0cm (norm_off=3.3) — non-overlap coverage for the new width
- w=3cm, off=10.0cm (norm_off=6.7) — large-offset anchor for w=3cm

Results saved to `openmc_data/gap_results.json` on Drive.

In [ ]:
import openmc, os, shutil, glob, json
import numpy as np

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
DET_VOLUME       = 200.0 * 100.0 * 22.0
GAP_RESULTS_FILE = '/content/drive/MyDrive/openmc_data/gap_results.json'

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

GAP_CASES = [
    (1.0,  7.5),   # w=1cm, off=7.5cm — brackets 1cm/10off in norm_off space
    (3.0,  5.0),   # w=3cm, off=5cm   — non-overlap for new width
    (3.0, 10.0),   # w=3cm, off=10cm  — large-offset anchor for w=3cm
]

if os.path.exists(GAP_RESULTS_FILE):
    with open(GAP_RESULTS_FILE) as f:
        results_gap = {tuple(map(float, k.split(','))): v for k, v in json.load(f).items()}
    print(f'Loaded {len(results_gap)} existing results from Drive.')
else:
    results_gap = {}

for gap_w, step_off in GAP_CASES:
    key = (gap_w, step_off)
    if key in results_gap:
        mean, std = results_gap[key]
        print(f'SKIP  w={gap_w:.1f}cm off={step_off:.1f}cm  (already done: {mean:.4e})')
        continue

    run_dir = f'/content/run_gap_{gap_w}cm_{step_off}off'
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)

    w    = gap_w / 2.0
    xp1m = openmc.XPlane(x0=-w);            xp1p = openmc.XPlane(x0=w)
    xp2m = openmc.XPlane(x0=-w+step_off);   xp2p = openmc.XPlane(x0=w+step_off)
    xp3m = openmc.XPlane(x0=-w+2*step_off); xp3p = openmc.XPlane(x0=w+2*step_off)
    seg1   = +xp1m & -xp1p & +z0   & -z_hm
    seg2   = +xp2m & -xp2p & +z_hm & -z_ml
    seg3   = +xp3m & -xp3p & +z_ml & -z1
    in_gap = seg1 | seg2 | seg3
    ib     = +x_min & -x_max & +y_min & -y_max

    src_c = openmc.Cell(fill=None,   region=ib & +z_src & -z0)
    hfs_c = openmc.Cell(fill=tih2,   region=ib & +z0    & -z_hm & ~in_gap)
    mfs_c = openmc.Cell(fill=b4c,    region=ib & +z_hm  & -z_ml & ~in_gap)
    lfs_c = openmc.Cell(fill=fe1422, region=ib & +z_ml  & -z1   & ~in_gap)
    gap_c = openmc.Cell(fill=void,   region=ib & in_gap)
    det_c = openmc.Cell(fill=None,   region=ib & +z1    & -z_det)

    openmc.Geometry(openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])).export_to_xml(path=f'{run_dir}/geometry.xml')
    materials.export_to_xml(path=f'{run_dir}/materials.xml')
    settings.export_to_xml(path=f'{run_dir}/settings.xml')

    ft = openmc.Tally(name='detector_total_flux')
    ft.filters = [openmc.CellFilter([det_c])]
    ft.scores  = ['flux']
    openmc.Tallies([ft]).export_to_xml(path=f'{run_dir}/tallies.xml')

    os.chdir(run_dir)
    openmc.run(threads=2, output=False)

    sp_file = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
    sp = openmc.StatePoint(sp_file)
    t  = sp.get_tally(name='detector_total_flux')
    mean, std = float(t.mean.flat[0]), float(t.std_dev.flat[0])
    results_gap[key] = [mean, std]

    with open(GAP_RESULTS_FILE, 'w') as f:
        json.dump({f'{k[0]},{k[1]}': v for k, v in results_gap.items()}, f, indent=2)

    print(f'w={gap_w:.1f}cm off={step_off:.1f}cm  flux={mean:.4e} +/- {std:.2e}  ({std/mean:.1%})  [saved]')

cols = f"{'Gap (cm)':>9} {'Offset (cm)':>12} {'Vol-int (n*cm)':>16} {'Avg flux (n/cm2)':>18} {'Rel err':>8}"
print(cols)
print('-' * 68)
for (gap_w, step_off), (mean, std) in sorted(results_gap.items()):
    print(f'{gap_w:>9.1f} {step_off:>12.1f} {mean:>16.4e} {mean/DET_VOLUME:>18.4e} {std/mean:>7.1%}')
print('Done. Re-run Section 11 to include these points.')


---
## Notes / TODO

1. **Fe-1422 material** â€” confirm exact composition and density from Table II of the paper.  
   Current placeholder is 316SS; update `fe1422` cell if STARFIRE used a different alloy.

2. **Source spectrum** â€” Table I may show a multi-group spectrum, not pure 14.1 MeV.  
   If so, replace `openmc.stats.Discrete` with `openmc.stats.Tabular(E_bins, probs)`.

3. **Dose normalization** â€” `detector_dose` tally needs normalization to rem/h per source n/cmÂ²/s  
   to match paper figures. Multiply by source intensity; divide by detector cell volume.

4. **Variance reduction** â€” 108 cm deep penetration needs weight windows or CADIS.  
   Activate `openmc.WeightWindows` if detector statistics are poor (rel. error > 20%).

5. **Hâ‚‚O layer** â€” check Fig. 2; if the paper includes a water layer, add a `water` cell  
   between two of the existing shield layers.

6. **Geometry plot axes** â€” `universe.plot()` with `axes=` requires OpenMC >= 0.13.3.  
   If it raises TypeError, remove the `axes=` arguments and call plt.subplot() separately.

7. **Phase 5** â€” once cases match, replace manual loop with a parametric sweep  
   (gap width Ã— offset) to generate ML training data.

---
## 13 — M5: Variance Reduction (Weight Windows / MAGIC)

Goal: push the sampling floor ~5 decades so stepped and solid cases converge, then report the streaming-reduction ratio with a real FSD.

**Order of execution (mandatory):**
- 13-A: load helpers (run once after sections 1-4)
- 13-B/C/D: Step 0 — straight 1 cm guardrail (paste 13-D output before continuing)
- 13-E/F/G: Step 1 — solid baseline (paste 13-G output)
- 13-H/I: Step 2 — stepped 1 cm / offset 5 cm
- 13-J: Step 3 — gate close (paste output)

**Requires:** sections 1-4 already run → globals `HFS_CM, MFS_CM, LFS_CM, TOTAL_CM, SLAB_HALF_WIDTH, hfs_mat, mfs_mat, lfs_mat, void, materials, src_E_eV, src_w`

**Drive checkpoint:** each gen cell copies `weight_windows.h5` to Drive. If Colab disconnects mid-run, skip the gen cell and load from Drive in the prod cell.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  13-A  M5 helper functions (run once after sections 1-4)    ║
# ╚══════════════════════════════════════════════════════════════╝
import os, glob, shutil, time
import numpy as np
import openmc

# ── geometry factories ────────────────────────────────────────────

def _m5_planes():
    """Shared axial + lateral boundary planes (new objects each call)."""
    z_src = openmc.ZPlane(z0=-20.0, boundary_type='vacuum')
    z0    = openmc.ZPlane(z0=0.0)
    z_hm  = openmc.ZPlane(z0=HFS_CM)
    z_ml  = openmc.ZPlane(z0=HFS_CM + MFS_CM)
    z1    = openmc.ZPlane(z0=TOTAL_CM)
    z_det = openmc.ZPlane(z0=TOTAL_CM + 52.0, boundary_type='vacuum')
    x_min = openmc.XPlane(x0=-SLAB_HALF_WIDTH, boundary_type='reflective')
    x_max = openmc.XPlane(x0= SLAB_HALF_WIDTH, boundary_type='reflective')
    y_min = openmc.YPlane(y0=-50.0, boundary_type='reflective')
    y_max = openmc.YPlane(y0= 50.0, boundary_type='reflective')
    in_box = +x_min & -x_max & +y_min & -y_max
    return z_src, z0, z_hm, z_ml, z1, z_det, in_box

def m5_make_geometry(gap_type, gap_width_cm, step_offset_cm=0.0):
    """Build STARFIRE geometry — matches section 3 exactly."""
    w   = gap_width_cm / 2.0
    off = step_offset_cm
    z_src, z0, z_hm, z_ml, z1, z_det, in_box = _m5_planes()

    if gap_type == 'straight':
        xgm    = openmc.XPlane(x0=-w)
        xgp    = openmc.XPlane(x0= w)
        in_gap = +xgm & -xgp & +z0 & -z1
    elif gap_type == 'stepped':
        xp1m = openmc.XPlane(x0=-w);         xp1p = openmc.XPlane(x0=w)
        xp2m = openmc.XPlane(x0=-w + off);   xp2p = openmc.XPlane(x0=w + off)
        xp3m = openmc.XPlane(x0=-w + 2*off); xp3p = openmc.XPlane(x0=w + 2*off)
        in_gap = ((+xp1m & -xp1p & +z0  & -z_hm) |
                  (+xp2m & -xp2p & +z_hm & -z_ml) |
                  (+xp3m & -xp3p & +z_ml & -z1))
    else:
        raise ValueError(f"gap_type must be 'straight' or 'stepped', got '{gap_type}'")

    src_c = openmc.Cell(name='source_void',   fill=None,    region=in_box & +z_src & -z0)
    hfs_c = openmc.Cell(name='HFS',           fill=hfs_mat, region=in_box & +z0   & -z_hm & ~in_gap)
    mfs_c = openmc.Cell(name='MFS',           fill=mfs_mat, region=in_box & +z_hm & -z_ml & ~in_gap)
    lfs_c = openmc.Cell(name='LFS',           fill=lfs_mat, region=in_box & +z_ml & -z1   & ~in_gap)
    gap_c = openmc.Cell(name='gap',           fill=void,    region=in_box & in_gap)
    det_c = openmc.Cell(name='detector_void', fill=None,    region=in_box & +z1   & -z_det)
    univ  = openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, gap_c, det_c])
    return openmc.Geometry(univ), det_c

def m5_make_solid_geometry():
    """Build solid STARFIRE shield — no gap."""
    z_src, z0, z_hm, z_ml, z1, z_det, in_box = _m5_planes()
    src_c = openmc.Cell(name='source_void',   fill=None,    region=in_box & +z_src & -z0)
    hfs_c = openmc.Cell(name='HFS',           fill=hfs_mat, region=in_box & +z0   & -z_hm)
    mfs_c = openmc.Cell(name='MFS',           fill=mfs_mat, region=in_box & +z_hm & -z_ml)
    lfs_c = openmc.Cell(name='LFS',           fill=lfs_mat, region=in_box & +z_ml & -z1)
    det_c = openmc.Cell(name='detector_void', fill=None,    region=in_box & +z1   & -z_det)
    univ  = openmc.Universe(cells=[src_c, hfs_c, mfs_c, lfs_c, det_c])
    return openmc.Geometry(univ), det_c

# ── tallies factory ───────────────────────────────────────────────

def m5_make_tallies(det_cell):
    """ICRP-116 dose mesh tallies + total flux in detector cell."""
    dose_E, dose_C = openmc.data.dose_coefficients('neutron', 'AP')
    dose_fn = openmc.EnergyFunctionFilter(dose_E, dose_C)

    def _mesh(z_lo, z_hi):
        m = openmc.RegularMesh()
        m.lower_left  = [-SLAB_HALF_WIDTH, -50.0, z_lo]
        m.upper_right = [ SLAB_HALF_WIDTH,  50.0, z_hi]
        m.dimension   = [20, 1, 1]
        return m

    def _pair(tag, mesh):
        fl = openmc.Tally(name=f'flux_{tag}')
        fl.filters = [openmc.MeshFilter(mesh)]
        fl.scores  = ['flux']
        do = openmc.Tally(name=f'dose_{tag}')
        do.filters = [openmc.MeshFilter(mesh), dose_fn]
        do.scores  = ['flux']
        return fl, do

    fl_s, do_s = _pair('surface', _mesh(TOTAL_CM,        TOTAL_CM + 1.0))
    fl_f, do_f = _pair('50cm',    _mesh(TOTAL_CM + 49.5, TOTAL_CM + 50.5))
    det_t = openmc.Tally(name='detector_total_flux')
    det_t.filters = [openmc.CellFilter([det_cell])]
    det_t.scores  = ['flux']
    return openmc.Tallies([fl_s, do_s, fl_f, do_f, det_t])

# ── source factory (reuses section-4 globals src_E_eV, src_w) ────

def m5_make_source():
    src = openmc.IndependentSource()
    src.space  = openmc.stats.Box(
        lower_left  = [-SLAB_HALF_WIDTH, -50.0, -10.0],
        upper_right = [ SLAB_HALF_WIDTH,  50.0,  -5.0],
    )
    src.angle  = openmc.stats.PolarAzimuthal(
        mu=openmc.stats.Uniform(0.0, 1.0),
        phi=openmc.stats.Uniform(0.0, 2.0 * np.pi),
    )
    src.energy = openmc.stats.Discrete(src_E_eV, src_w)
    return src

# ── WW mesh (shared by all generation passes) ─────────────────────

def _ww_mesh():
    """10 cm x-bins, 2 cm z-bins, single y-bin — 1 600 cells total."""
    m = openmc.RegularMesh()
    m.lower_left  = [-100.0, -50.0, -20.0]
    m.upper_right = [ 100.0,  50.0, 160.0]
    m.dimension   = [20, 1, 90]
    return m

# ── export + run helper ───────────────────────────────────────────

def m5_run(run_dir, geom, tallies, settings):
    """Export XMLs, chdir, run, return statepoint path."""
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)
    materials.export_to_xml(path=f'{run_dir}/materials.xml')
    geom.export_to_xml(path=f'{run_dir}/geometry.xml')
    tallies.export_to_xml(path=f'{run_dir}/tallies.xml')
    settings.export_to_xml(path=f'{run_dir}/settings.xml')
    os.chdir(run_dir)
    t0 = time.time()
    openmc.run(threads=2, output=True)
    elapsed = time.time() - t0
    sp = sorted(glob.glob(f'{run_dir}/statepoint.*.h5'))[-1]
    print(f"Done in {elapsed/60:.1f} min  |  {sp}")
    return sp

DRIVE_DATA = '/content/drive/MyDrive/openmc_data'
os.makedirs(DRIVE_DATA, exist_ok=True)
print("13-A ready.  Helpers: m5_make_geometry, m5_make_solid_geometry, m5_make_tallies, m5_run")

In [ ]:
# ── 13-B  Step 0-GEN  straight 1 cm — build importance map (~15 min) ──
WW_DIR_S = '/content/run_m5ww_straight1_gen'

geom_s, det_s = m5_make_geometry('straight', 1.0)
tals_s = m5_make_tallies(det_s)

sets_sg = openmc.Settings()
sets_sg.source    = m5_make_source()
sets_sg.run_mode  = 'fixed source'
sets_sg.batches   = 40
sets_sg.particles = 20_000
sets_sg.weight_window_generators = [openmc.WeightWindowGenerator(
    method='magic',
    mesh=_ww_mesh(),
    energy_bounds=[0.0, 14.1e6],
    particle_type='neutron',
    max_realizations=20,
    update_interval=1,
    on_the_fly=True,
)]

m5_run(WW_DIR_S, geom_s, tals_s, sets_sg)

WW_H5_S = f'{WW_DIR_S}/weight_windows.h5'
if os.path.exists(WW_H5_S):
    shutil.copy(WW_H5_S, f'{DRIVE_DATA}/ww_straight1cm.h5')
    print("Checkpointed -> Drive: ww_straight1cm.h5")
else:
    print("WARNING: weight_windows.h5 not found — check output above for errors")

In [ ]:
# ── 13-C  Step 0-PROD  straight 1 cm production run (~25 min) ────
PROD_DIR_S = '/content/run_m5ww_straight1_prod'

geom_sp, det_sp = m5_make_geometry('straight', 1.0)
tals_sp = m5_make_tallies(det_sp)

sets_sp = openmc.Settings()
sets_sp.source    = m5_make_source()
sets_sp.run_mode  = 'fixed source'
sets_sp.batches   = 200
sets_sp.particles = 50_000

ww_h5 = f'{DRIVE_DATA}/ww_straight1cm.h5'
if not os.path.exists(ww_h5):
    ww_h5 = f'{WW_DIR_S}/weight_windows.h5'   # local fallback if Drive unavailable
wws_s = openmc.WeightWindowsList.from_hdf5(ww_h5)
sets_sp.weight_windows           = wws_s
sets_sp.weight_windows_on        = True
sets_sp.weight_window_generators = []

m5_run(PROD_DIR_S, geom_sp, tals_sp, sets_sp)

In [ ]:
# ── 13-D  Step 0-CHECK  guardrail — PASTE THIS OUTPUT BEFORE CONTINUING ──
ANALOG_S1 = 2.32e-9    # M2 validated result

sp_path = sorted(glob.glob(f'{PROD_DIR_S}/statepoint.*.h5'))[-1]
print(f"Statepoint: {sp_path}")
with openmc.StatePoint(sp_path) as sp:
    t = sp.get_tally(name='detector_total_flux')
    s_mean = float(t.mean.flat[0])
    s_std  = float(t.std_dev.flat[0])

s_fsd = s_std / s_mean if s_mean > 0 else float('nan')
ratio = s_mean / ANALOG_S1

print(f"\n{'='*55}")
print(f"  STEP 0 — Straight 1 cm guardrail")
print(f"{'='*55}")
print(f"  WW result  : {s_mean:.4e} +/- {s_std:.2e}  FSD={s_fsd:.3f}")
print(f"  Analog (M2): {ANALOG_S1:.4e}")
print(f"  Ratio      : {ratio:.3f}   (pass if |ratio-1| <= 3*FSD = {3*s_fsd:.2f})")

if abs(ratio - 1.0) <= 3 * s_fsd:
    print("\n  GUARDRAIL PASSED — WW is unbiased; proceed to solid / stepped")
else:
    print("\n  GUARDRAIL FAILED — stop here")
    print("    Fix: coarsen WW mesh (dim [10,1,45]) or raise gen particles to 50k")

In [ ]:
# ── 13-E  Step 1-GEN  solid baseline — deepest case, more iterations (~25 min) ──
WW_DIR_SOL = '/content/run_m5ww_solid_gen'

geom_sol, det_sol = m5_make_solid_geometry()
tals_sol = m5_make_tallies(det_sol)

sets_solg = openmc.Settings()
sets_solg.source    = m5_make_source()
sets_solg.run_mode  = 'fixed source'
sets_solg.batches   = 60
sets_solg.particles = 20_000
sets_solg.weight_window_generators = [openmc.WeightWindowGenerator(
    method='magic',
    mesh=_ww_mesh(),
    energy_bounds=[0.0, 14.1e6],
    particle_type='neutron',
    max_realizations=30,    # deeper penetration needs more MAGIC iterations
    update_interval=1,
    on_the_fly=True,
)]

m5_run(WW_DIR_SOL, geom_sol, tals_sol, sets_solg)

WW_H5_SOL = f'{WW_DIR_SOL}/weight_windows.h5'
if os.path.exists(WW_H5_SOL):
    shutil.copy(WW_H5_SOL, f'{DRIVE_DATA}/ww_solid.h5')
    print("Checkpointed -> Drive: ww_solid.h5")
else:
    print("WARNING: weight_windows.h5 not found")

In [ ]:
# ── 13-F  Step 1-PROD  solid production run (~25 min) ────────────
PROD_DIR_SOL = '/content/run_m5ww_solid_prod'

geom_solp, det_solp = m5_make_solid_geometry()
tals_solp = m5_make_tallies(det_solp)

sets_solp = openmc.Settings()
sets_solp.source    = m5_make_source()
sets_solp.run_mode  = 'fixed source'
sets_solp.batches   = 200
sets_solp.particles = 50_000

ww_h5 = f'{DRIVE_DATA}/ww_solid.h5'
if not os.path.exists(ww_h5):
    ww_h5 = f'{WW_DIR_SOL}/weight_windows.h5'
wws_sol = openmc.WeightWindowsList.from_hdf5(ww_h5)
sets_solp.weight_windows           = wws_sol
sets_solp.weight_windows_on        = True
sets_solp.weight_window_generators = []

m5_run(PROD_DIR_SOL, geom_solp, tals_solp, sets_solp)

In [ ]:
# ── 13-G  Step 1-READ  solid baseline result — PASTE THIS OUTPUT ──
sp_path = sorted(glob.glob(f'{PROD_DIR_SOL}/statepoint.*.h5'))[-1]
print(f"Statepoint: {sp_path}")
with openmc.StatePoint(sp_path) as sp:
    t = sp.get_tally(name='detector_total_flux')
    sol_mean = float(t.mean.flat[0])
    sol_std  = float(t.std_dev.flat[0])

sol_fsd = sol_std / sol_mean if sol_mean > 0 else float('nan')

print(f"\n{'='*55}")
print(f"  STEP 1 — Solid baseline")
print(f"{'='*55}")
print(f"  flux = {sol_mean:.4e} +/- {sol_std:.2e}   FSD={sol_fsd:.3f}")

if sol_fsd < 0.10:
    print("  Converged  (FSD < 0.10)")
elif sol_fsd < 0.30:
    print("  Partially converged (FSD < 0.30) — OK for bounding ratio")
else:
    print("  Not converged — escalate: need more MAGIC iterations or energy groups")

In [ ]:
# ── 13-H  Step 2-GEN  stepped 1 cm / offset 5 cm — headline case (~20 min) ──
WW_DIR_ST5 = '/content/run_m5ww_stepped1_off5_gen'

geom_st5, det_st5 = m5_make_geometry('stepped', 1.0, step_offset_cm=5.0)
tals_st5 = m5_make_tallies(det_st5)

sets_st5g = openmc.Settings()
sets_st5g.source    = m5_make_source()
sets_st5g.run_mode  = 'fixed source'
sets_st5g.batches   = 50
sets_st5g.particles = 20_000
sets_st5g.weight_window_generators = [openmc.WeightWindowGenerator(
    method='magic',
    mesh=_ww_mesh(),
    energy_bounds=[0.0, 14.1e6],
    particle_type='neutron',
    max_realizations=25,
    update_interval=1,
    on_the_fly=True,
)]

m5_run(WW_DIR_ST5, geom_st5, tals_st5, sets_st5g)

WW_H5_ST5 = f'{WW_DIR_ST5}/weight_windows.h5'
if os.path.exists(WW_H5_ST5):
    shutil.copy(WW_H5_ST5, f'{DRIVE_DATA}/ww_stepped1_off5.h5')
    print("Checkpointed -> Drive: ww_stepped1_off5.h5")
else:
    print("WARNING: weight_windows.h5 not found")

In [ ]:
# ── 13-I  Step 2-PROD  stepped 1 cm / offset 5 cm production (~25 min) ──
PROD_DIR_ST5 = '/content/run_m5ww_stepped1_off5_prod'

geom_st5p, det_st5p = m5_make_geometry('stepped', 1.0, step_offset_cm=5.0)
tals_st5p = m5_make_tallies(det_st5p)

sets_st5p = openmc.Settings()
sets_st5p.source    = m5_make_source()
sets_st5p.run_mode  = 'fixed source'
sets_st5p.batches   = 200
sets_st5p.particles = 50_000

ww_h5 = f'{DRIVE_DATA}/ww_stepped1_off5.h5'
if not os.path.exists(ww_h5):
    ww_h5 = f'{WW_DIR_ST5}/weight_windows.h5'
wws_st5 = openmc.WeightWindowsList.from_hdf5(ww_h5)
sets_st5p.weight_windows           = wws_st5
sets_st5p.weight_windows_on        = True
sets_st5p.weight_window_generators = []

m5_run(PROD_DIR_ST5, geom_st5p, tals_st5p, sets_st5p)

In [ ]:
# ── 13-J  Step 3 — Gate Close  PASTE THIS OUTPUT ─────────────────
ANALOG_S1 = 2.32e-9   # M2 validated (re-declare in case kernel was restarted)

sp_path = sorted(glob.glob(f'{PROD_DIR_ST5}/statepoint.*.h5'))[-1]
print(f"Statepoint: {sp_path}")
with openmc.StatePoint(sp_path) as sp:
    t = sp.get_tally(name='detector_total_flux')
    st5_mean = float(t.mean.flat[0])
    st5_std  = float(t.std_dev.flat[0])

st5_fsd  = st5_std / st5_mean if st5_mean > 0 else float('nan')
ratio_ss = st5_mean / ANALOG_S1
ratio_so = st5_mean / sol_mean if sol_mean > 0 else float('nan')
dec_ss   = -np.log10(ratio_ss) if ratio_ss > 0 else float('nan')
dec_so   = -np.log10(ratio_so) if (ratio_so > 0 and not np.isnan(ratio_so)) else float('nan')

print(f"\n{'='*60}")
print(f"  M5 RESULTS — Gate Close")
print(f"{'='*60}")
print(f"  Straight 1cm (M2 analog):  {ANALOG_S1:.4e} n/cm2")
print(f"  Solid (WW):                {sol_mean:.4e} +/- {sol_std:.2e}   FSD={sol_fsd:.3f}")
print(f"  Stepped 1cm/off5 (WW):     {st5_mean:.4e} +/- {st5_std:.2e}   FSD={st5_fsd:.3f}")
print(f"\n  Streaming reduction:")
print(f"    stepped / straight : {ratio_ss:.3e}   ({dec_ss:.1f} decades)  [paper target ~5]")
print(f"    stepped / solid    : {ratio_so:.3e}   ({dec_so:.1f} decades  streaming enhancement)")

if st5_fsd < 0.10:
    print(f"\n  Stepped converged (FSD={st5_fsd:.3f} < 0.10)")
elif st5_fsd < 0.20:
    print(f"\n  Marginal convergence (FSD={st5_fsd:.3f})")
else:
    print(f"\n  Not converged (FSD={st5_fsd:.3f}) — more batches or MAGIC iterations")

if not np.isnan(dec_ss):
    if dec_ss >= 4.5:
        print(f"  GATE CLOSED ({dec_ss:.1f} decades >= 4.5 — consistent with paper ~5)")
    elif dec_ss >= 3.0:
        print(f"  Gate partially closed ({dec_ss:.1f} decades) — report as lower bound")
    else:
        print(f"  {dec_ss:.1f} decades only — escalate to Opus")

print(f"\n  --- FOM ---")
print(f"  FOM = 1 / (FSD^2 * T_seconds)")
print(f"  Paste run times from 13-C and 13-I to compute FOM_WW / FOM_analog speedup")

---
## 14 — M5b: single-step dog-leg (Fig IV-8)
Reproduce the 5-decade centerline reduction for a single ~3 cm step.  
Additive — does not modify validated M1/M2/M5 cells.


In [ ]:
# ─── 14-A  Config overrides for single_step ─────────────────────────────
import os
GAP_TYPE       = "single_step"
GAP_WIDTH_CM   = 1.0
STEP_OFFSET_CM = 3.0           # Fig IV-8 centre-to-centre offset (cm)
RUN_DIR        = f"/content/run_{GAP_TYPE}_{GAP_WIDTH_CM}cm"
os.makedirs(RUN_DIR, exist_ok=True)
print(f"Output: {RUN_DIR}")
print(f"Gap: {GAP_TYPE}, width={GAP_WIDTH_CM} cm, offset={STEP_OFFSET_CM} cm")
# After running this cell, re-run cell 2 (Materials) and cell 3 (Geometry)


In [ ]:
# ─── 14-B  Fine centerline dose tallies (0.5 cm bins, surface + 50 cm) ──
import openmc, numpy as np

dose_E, dose_C = openmc.data.dose_coefficients("neutron", "AP")
dose_fn = openmc.EnergyFunctionFilter(dose_E, dose_C)

def plane_mesh(z_lo, z_hi):
    m = openmc.RegularMesh()
    m.lower_left  = [-10.0, -50.0, z_lo]
    m.upper_right = [ 10.0,  50.0, z_hi]
    m.dimension   = [40, 1, 1]   # 0.5 cm x-bins
    return m

surf_mesh = plane_mesh(TOTAL_CM,        TOTAL_CM + 1.0)      # z=108 (surface)
far_mesh  = plane_mesh(TOTAL_CM + 49.5, TOTAL_CM + 50.5)     # z=158 (+50 cm, Fig IV-8)

def dose_tally(tag, mesh):
    t = openmc.Tally(name=f"dose_{tag}")
    t.filters = [openmc.MeshFilter(mesh), dose_fn]
    t.scores  = ["flux"]
    return t

tallies = openmc.Tallies([
    dose_tally("surface", surf_mesh),
    dose_tally("50cm",    far_mesh),
    openmc.Tally(name="detector_total_flux"),
])
tallies[-1].filters = [openmc.CellFilter([det_cell])]
tallies[-1].scores  = ["flux"]
tallies.export_to_xml(path=f"{RUN_DIR}/tallies.xml")
print("Tallies written:", [t.name for t in tallies])


In [ ]:
# ─── 14-C  Refined WW generation (fine x near gap, MAGIC) ─────────────
import openmc, numpy as np, os, shutil

DRIVE = "/content/drive/MyDrive/openmc_data"
WW_GEN_DIR = "/content/ww_gen_single_step"
os.makedirs(WW_GEN_DIR, exist_ok=True)
# Copy geometry + materials into WW gen dir
for f in ["geometry.xml", "materials.xml", "tallies.xml"]:
    shutil.copy(f"{RUN_DIR}/{f}", f"{WW_GEN_DIR}/{f}")

x_grid = np.unique(np.concatenate([
    np.linspace(-100, -10, 10),   # coarse outer
    np.linspace( -10,  10, 41),   # fine 0.5 cm (gap + shadow + redirected leg)
    np.linspace(  10, 100, 10),   # coarse outer
]))
ww_mesh = openmc.RectilinearMesh()
ww_mesh.x_grid = x_grid
ww_mesh.y_grid = np.array([-50.0, 50.0])
ww_mesh.z_grid = np.linspace(-20.0, 160.0, 91)   # 2 cm in z

wwg = openmc.WeightWindowGenerator(
    method           = "magic",
    mesh             = ww_mesh,
    energy_bounds    = [0.0, 14.1e6],
    particle_type    = "neutron",
    max_realizations = 40,
    update_interval  = 1,
    on_the_fly       = True,
)

sets_ww = openmc.Settings()
sets_ww.run_mode                = "fixed source"
sets_ww.particles               = 20_000
sets_ww.batches                 = 40
sets_ww.weight_windows_on       = True
sets_ww.weight_window_generators = [wwg]
sets_ww.export_to_xml(path=f"{WW_GEN_DIR}/settings.xml")

openmc.run(cwd=WW_GEN_DIR)

ww_dst = f"{DRIVE}/ww_single_step_off3.h5"
shutil.copy(f"{WW_GEN_DIR}/weight_windows.h5", ww_dst)
print(f"Weight windows saved: {ww_dst}")


In [ ]:
# ─── 14-D  Production run + Fig IV-8 centerline comparison ───────────
import openmc, numpy as np, glob, shutil, os

DRIVE       = "/content/drive/MyDrive/openmc_data"
PROD_DIR_SS = "/content/prod_single_step_1cm"
os.makedirs(PROD_DIR_SS, exist_ok=True)
for f in ["geometry.xml", "materials.xml", "tallies.xml"]:
    shutil.copy(f"{RUN_DIR}/{f}", f"{PROD_DIR_SS}/{f}")

# Load saved weight windows
wws_ss = openmc.WeightWindowsList.from_hdf5(f"{DRIVE}/ww_single_step_off3.h5")

sets_ss = openmc.Settings()
sets_ss.run_mode          = "fixed source"
sets_ss.particles         = 50_000
sets_ss.batches           = 200
sets_ss.weight_windows    = wws_ss
sets_ss.weight_windows_on = True
sets_ss.export_to_xml(path=f"{PROD_DIR_SS}/settings.xml")
openmc.run(cwd=PROD_DIR_SS)

sp_ss_path = sorted(glob.glob(f"{PROD_DIR_SS}/statepoint.*.h5"))[-1]
print("single_step statepoint:", sp_ss_path)
shutil.copy(sp_ss_path, f"{DRIVE}/sp_single_step_off3.h5")

# ── Analysis ──────────────────────────────────────────────────────────
x_edges = np.linspace(-10, 10, 41)
xc = 0.5 * (x_edges[:-1] + x_edges[1:])
i0 = int(np.argmin(np.abs(xc)))

def centerline(path, name="dose_50cm"):
    sp  = openmc.StatePoint(path)
    t   = sp.get_tally(name=name)
    m   = t.mean.flatten()
    s   = t.std_dev.flatten()
    rel = np.divide(s, m, out=np.full_like(m, np.inf), where=m > 0)
    ipk = int(np.argmax(m))
    return dict(x0=m[i0], x0_rel=rel[i0], peak=m[ipk], peak_x=xc[ipk], peak_rel=rel[ipk])

# Straight reuses M5 fine statepoint
sp_straight = f"{DRIVE}/sp_straight1cm_fine.h5"
st = centerline(sp_straight)
ss = centerline(sp_ss_path)

print(f"straight  : x=0 dose={st['x0']:.3e}  (rel {st['x0_rel']:.2f})  "
      f"peak={st['peak']:.3e} @x={st['peak_x']:.1f}")
print(f"singlestep: x=0 dose={ss['x0']:.3e}  (rel {ss['x0_rel']:.2f})  "
      f"peak={ss['peak']:.3e} @x={ss['peak_x']:.1f}  (rel {ss['peak_rel']:.2f})")

if ss["x0"] > 0 and st["x0"] > 0:
    ratio = st["x0"] / ss["x0"]
    print(f"\nx=0 reduction  straight/singlestep = {ratio:.3e}  "
          f"= {np.log10(ratio):.2f} decades   (Fig IV-8 target ~5)")
else:
    print("WARNING: single_step x=0 bin is zero/censored -- WW did not reach the shadow.")
    print("Action: switch 14-C to method='fw_cadis' and re-run 14-C + 14-D.")

peak_ratio = st["peak"] / ss["peak"]
print(f"redirected peak vs straight peak  = {peak_ratio:.2f}x   (Fig IV-8 target ~4x)")
